<a href="https://colab.research.google.com/github/yangyi02/droid/blob/main/PointWorld_compute_depth_and_extrinsics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
## 🚀 Colab One-Time Setup

Run **all cells in this section once** per runtime. They clone the repo, install dependencies, and download model weights.

> **Requirements**: Use a **GPU runtime** (T4 or better). Go to `Runtime → Change runtime type → GPU`.

In [ ]:
import os

# Clone PointWorld repo with all submodules (switch to 'data' branch)
COLAB_ROOT = "/content/PointWorld"
if not os.path.exists(COLAB_ROOT):
    !git clone --recurse-submodules https://github.com/NVlabs/PointWorld.git {COLAB_ROOT}
    %cd {COLAB_ROOT}
    !git checkout data
    !git submodule update --init --recursive
else:
    %cd {COLAB_ROOT}
    print(f"PointWorld already cloned at {COLAB_ROOT}")

print(f"Working directory: {os.getcwd()}")

In [ ]:
# Install all required Python packages
# Step 1: Upgrade numpy to 2.x first (required by opencv, jax, etc. in Colab)
!pip install -q "numpy>=2.0,<2.3"

# Step 2: Install numba (latest, supports numpy 2.x)
# Do NOT pin numba==0.62.0 — it was compiled against numpy 1.x ABI
!pip install -q numba

# Step 3: Install remaining dependencies
!pip install -q \
    urdfpy \
    omegaconf \
    "timm==0.9.16" \
    einops \
    trimesh \
    h5py \
    tqdm \
    scipy \
    scikit-learn \
    scikit-image \
    transforms3d \
    transformations \
    huggingface_hub \
    safetensors \
    opencv-contrib-python \
    pytorch_kinematics \
    open3d \
    pyyaml

# Install VGGT from the submodule
!pip install -q {COLAB_ROOT}/third_party/vggt

# Verify numpy and numba versions are compatible
import numpy as np
import numba
print(f"\n✅ All Python dependencies installed.")
print(f"   numpy  : {np.__version__}")
print(f"   numba  : {numba.__version__}")

In [ ]:
# Install ZED SDK for SVO file decoding
import os

ZED_SDK_INSTALLED = False
try:
    import pyzed.sl as sl
    ZED_SDK_INSTALLED = True
    print("✅ ZED SDK (pyzed) already installed.")
except ImportError:
    print("ZED SDK not found. Installing...")
    # Download and install the ZED SDK (headless, for Colab/server environments)
    !wget -q --show-progress -O /tmp/zed_sdk.run \
        "https://download.stereolabs.com/zedsdk/4.2/cu121/ubuntu22" && \
        chmod +x /tmp/zed_sdk.run && \
        /tmp/zed_sdk.run -- silent skip_cuda skip_od_module skip_tools && \
        rm /tmp/zed_sdk.run
    # Install the Python bindings
    !pip install -q /usr/local/zed/pyzed*.whl 2>/dev/null || \
        python /usr/local/zed/get_python_api.py
    try:
        import pyzed.sl as sl
        ZED_SDK_INSTALLED = True
        print("✅ ZED SDK installed successfully.")
    except ImportError:
        print("⚠️ ZED SDK installation failed. SVO decoding will not work.")
        print("   You can try installing manually: https://www.stereolabs.com/docs/installation/linux")

In [ ]:
import os

# Download FoundationStereo checkpoint from GCS
CKPT_DIR = os.path.join(COLAB_ROOT, "checkpoints", "foundationstereo", "23-51-11")
CKPT_PATH = os.path.join(CKPT_DIR, "model_best_bp2.pth")

if not os.path.exists(CKPT_PATH):
    os.makedirs(CKPT_DIR, exist_ok=True)
    # Download from GCS (requires authentication)
    !gcloud storage cp gs://dm-tapnet/checkpoints/foundationstereo/23-51-11/model_best_bp2.pth {CKPT_PATH}
    print(f"✅ FoundationStereo checkpoint downloaded to {CKPT_PATH}")
else:
    print(f"⏭️ FoundationStereo checkpoint already exists at {CKPT_PATH}")

# Verify file size (model should be > 100MB)
if os.path.exists(CKPT_PATH):
    size_mb = os.path.getsize(CKPT_PATH) / (1024 * 1024)
    print(f"   File size: {size_mb:.1f} MB")

In [ ]:
import os

# Download VGGT checkpoint from HuggingFace
VGGT_DIR = os.path.join(COLAB_ROOT, "checkpoints", "vggt")
VGGT_PATH = os.path.join(VGGT_DIR, "model.pt")

if not os.path.exists(VGGT_PATH):
    os.makedirs(VGGT_DIR, exist_ok=True)
    from huggingface_hub import hf_hub_download
    downloaded = hf_hub_download(
        repo_id="facebook/VGGT-1B",
        filename="model.pt",
        local_dir=VGGT_DIR,
    )
    print(f"✅ VGGT checkpoint downloaded to {VGGT_PATH}")
else:
    print(f"⏭️ VGGT checkpoint already exists at {VGGT_PATH}")

# Verify
if os.path.exists(VGGT_PATH):
    size_mb = os.path.getsize(VGGT_PATH) / (1024 * 1024)
    print(f"   File size: {size_mb:.1f} MB")

In [ ]:
# Authenticate with Google Cloud for GCS access to DROID data
from google.colab import auth
auth.authenticate_user()

# Set GCS cache directory for DROID data
import os
os.environ['POINTWORLD_CACHE_DIR'] = '/content/pointworld_cache'
os.makedirs(os.environ['POINTWORLD_CACHE_DIR'], exist_ok=True)
print(f"✅ GCS auth complete. Cache dir: {os.environ['POINTWORLD_CACHE_DIR']}")

# Quick verification that GCS is accessible
!gcloud storage ls gs://gresearch/robotics/droid_raw/1.0.1/ 2>&1 | head -5

# Compute Depth & Extrinsics for PointWorld

This notebook runs the full pipeline for:

1. **Compute Depth** — Stereo depth estimation via FoundationStereo, saved to H5 files.
2. **Compute Extrinsics** — Camera extrinsics optimization via VGGT + robot mesh rendering.

> **How to use**: Run all cells top-to-bottom. The Colab Setup section above handles all dependencies automatically.

---
## Step 0 — Environment Setup

Set up `sys.path` so the written modules are importable, and create the required directory structure.

In [ ]:
import sys, os

# Point to the cloned PointWorld repo
REPO_ROOT = "/content/PointWorld"
REAL_DIR = os.path.join(REPO_ROOT, "real")

# Ensure repo root and real/ are importable
for p in [REPO_ROOT, REAL_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

# Also add VGGT to sys.path
VGGT_ROOT = os.path.join(REPO_ROOT, "third_party", "vggt")
if VGGT_ROOT not in sys.path:
    sys.path.append(VGGT_ROOT)

print("REPO_ROOT:", REPO_ROOT)
print("sys.path entries:", sys.path[:5])

---
## Step 1 — Define: `gcs_utils` (GCS helpers)

In [ ]:
# ============================================================
# gcs_utils — Google Cloud Storage helpers
# ============================================================
import os, re, tempfile, subprocess

def is_gcs_path(path):
    """Check if the path is a Google Cloud Storage path."""
    return path.startswith('gs://')

def parse_gcs_path(gcs_path):
    """Parse a GCS path into bucket and blob names."""
    match = re.match(r'gs://([^/]+)/(.*)', gcs_path)
    if not match:
        raise ValueError(f"Invalid GCS path: {gcs_path}")
    return match.group(1), match.group(2)

def download_from_gcs(gcs_path, local_path):
    """Download a file from GCS to a local path using gsutil."""
    os.makedirs(os.path.dirname(local_path), exist_ok=True)
    cmd = ["gsutil", "cp", gcs_path, local_path]
    try:
        subprocess.run(cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    except subprocess.CalledProcessError as e:
        raise RuntimeError(f"GCS download failed for {gcs_path}: {e.stderr.strip()}") from e
    return local_path

def get_local_path(path, temp_dir=None):
    """If GCS path, download to local cache or temp dir and return local path."""
    if not is_gcs_path(path):
        return path
    cache_root = os.environ.get('POINTWORLD_CACHE_DIR')
    if cache_root:
        bucket, blob = parse_gcs_path(path)
        local_path = os.path.join(cache_root, "droid", bucket, blob)
        os.makedirs(os.path.dirname(local_path), exist_ok=True)
        if os.path.exists(local_path):
            return local_path
        download_from_gcs(path, local_path)
        return local_path
    else:
        if temp_dir is None:
            temp_dir = tempfile.mkdtemp()
        local_path = os.path.join(temp_dir, os.path.basename(path))
        download_from_gcs(path, local_path)
        return local_path

def enforce_gcs_cache_policy(scene_paths, stage_name, require_cache=False, allow_streaming=False):
    """Validate cache policy for GCS-backed scene paths."""
    if isinstance(scene_paths, str):
        scene_paths = [scene_paths]
    gcs_paths = [p for p in scene_paths if is_gcs_path(p)]
    if not gcs_paths:
        return False
    cache_root = os.environ.get("POINTWORLD_CACHE_DIR", "").strip()
    if cache_root:
        print(f"[{stage_name}] cache enabled: POINTWORLD_CACHE_DIR={cache_root}")
        return True
    message = (
        f"[{stage_name}] Detected {len(gcs_paths)} GCS scene path(s) but POINTWORLD_CACHE_DIR is not set. "
        "Without persistent caching, repeated stages may re-download the same objects."
    )
    setup_hint = "Set: export POINTWORLD_CACHE_DIR=/path/to/fast_disk/pointworld_cache"
    if require_cache and not allow_streaming:
        raise RuntimeError(f"{message} {setup_hint}")
    print(f"WARNING: {message} Continuing because allow_streaming={allow_streaming}.")
    return True

def list_gcs_files(gcs_path, pattern=None):
    """List files in a GCS directory that match a pattern using gsutil."""
    if not gcs_path.endswith('/'):
        gcs_path += '/'
    search_path = os.path.join(gcs_path, pattern) if pattern else gcs_path + '*'
    cmd = ["gsutil", "ls", search_path]
    try:
        result = subprocess.run(cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        return [line.strip() for line in result.stdout.split('\n') if line.strip()]
    except subprocess.CalledProcessError as e:
        if "No URLs matched" in e.stderr:
            return []
        raise RuntimeError(f"Error listing files in {gcs_path}: {e.stderr}")

print("✅ gcs_utils defined")

---
## Step 2 — Define: `real_utils` (time helpers & numba kernels)

In [ ]:
# ============================================================
# real_utils — time helpers and numba projection kernels
# ============================================================
import numpy as np
from datetime import datetime

try:
    from numba import njit
    _NUMBA_AVAILABLE = True
except (ImportError, ValueError) as _numba_err:
    # ValueError can occur when numba C extensions were compiled against a
    # different numpy ABI (e.g. numpy 1.x vs 2.x). Fall back to a pure Python
    # stub so the rest of the notebook still works.
    print(f"⚠️ numba import failed ({_numba_err}); using pure-Python fallback for projection kernel.")
    _NUMBA_AVAILABLE = False
    def njit(*args, **kwargs):
        """No-op decorator when numba is unavailable."""
        def decorator(fn): return fn
        return decorator if args and callable(args[0]) else decorator

@njit(cache=True, fastmath=True, nogil=True)
def project_points_to_image(points_3d, transform_matrix, extrinsic, intrinsic, image_width, image_height):
    """Fused kernel to project 3D points to 2D image coordinates."""
    n_points = points_3d.shape[0]
    projected_points = np.empty((n_points, 2), dtype=np.float32)
    valid_count = 0
    for i in range(n_points):
        local_point = np.array([points_3d[i, 0], points_3d[i, 1], points_3d[i, 2], 1.0], dtype=np.float32)
        world_point = np.zeros(4, dtype=np.float32)
        for j in range(4):
            world_point[j] = (transform_matrix[j, 0] * local_point[0] +
                              transform_matrix[j, 1] * local_point[1] +
                              transform_matrix[j, 2] * local_point[2] +
                              transform_matrix[j, 3] * local_point[3])
        cam_point = np.zeros(4, dtype=np.float32)
        for j in range(4):
            cam_point[j] = (extrinsic[j, 0] * world_point[0] +
                            extrinsic[j, 1] * world_point[1] +
                            extrinsic[j, 2] * world_point[2] +
                            extrinsic[j, 3] * world_point[3])
        if cam_point[2] <= 0:
            continue
        img_x = (intrinsic[0, 0] * cam_point[0] + intrinsic[0, 2] * cam_point[2]) / cam_point[2]
        img_y = (intrinsic[1, 1] * cam_point[1] + intrinsic[1, 2] * cam_point[2]) / cam_point[2]
        if img_x >= 0 and img_x < image_width and img_y >= 0 and img_y < image_height:
            projected_points[valid_count, 0] = img_x
            projected_points[valid_count, 1] = img_y
            valid_count += 1
    return projected_points[:valid_count].copy()

def get_mesh_name(mesh, idx):
    try:
        return f'{mesh.source.file_name.lower()}_{idx}'
    except AttributeError:
        return f'{mesh.metadata.get("name", mesh.metadata.get("file_name", f"unknown")).lower()}_{idx}'

def get_time_str():
    """Return current time in the format: YYYY-MM-DD HH:MM:SS"""
    return datetime.now().strftime('%Y-%m-%d %H:%M:%S')

print(f"✅ real_utils defined (numba available: {_NUMBA_AVAILABLE})")

---
## Step 3 — Define: `transform_utils` (rotation / pose math)

In [ ]:
# ============================================================
# transform_utils — matrix and vector transformation utilities
# ============================================================
import math
import numpy as np
from scipy.spatial.transform import Rotation as R

# Numba import with fallback (same pattern as real_utils)
try:
    from numba import njit
    _TU_NUMBA_AVAILABLE = True
except (ImportError, ValueError) as _numba_err:
    _TU_NUMBA_AVAILABLE = False
    def njit(*args, **kwargs):
        def decorator(fn): return fn
        return decorator if args and callable(args[0]) else decorator

PI = np.pi
EPS = np.finfo(float).eps * 4.0

@njit(cache=True, fastmath=True)
def _mat2quat_single(Rmat):
    t = Rmat[0,0] + Rmat[1,1] + Rmat[2,2]
    if t > 0.0:
        s = 0.5 / np.sqrt(t + 1.0)
        w = 0.25 / s
        x = (Rmat[2,1] - Rmat[1,2]) * s
        y = (Rmat[0,2] - Rmat[2,0]) * s
        z = (Rmat[1,0] - Rmat[0,1]) * s
    else:
        if Rmat[0,0] > Rmat[1,1] and Rmat[0,0] > Rmat[2,2]:
            s = 2.0 * np.sqrt(1.0 + Rmat[0,0] - Rmat[1,1] - Rmat[2,2])
            w = (Rmat[2,1] - Rmat[1,2]) / s; x = 0.25 * s
            y = (Rmat[0,1] + Rmat[1,0]) / s; z = (Rmat[0,2] + Rmat[2,0]) / s
        elif Rmat[1,1] > Rmat[2,2]:
            s = 2.0 * np.sqrt(1.0 + Rmat[1,1] - Rmat[0,0] - Rmat[2,2])
            w = (Rmat[0,2] - Rmat[2,0]) / s; x = (Rmat[0,1] + Rmat[1,0]) / s
            y = 0.25 * s; z = (Rmat[1,2] + Rmat[2,1]) / s
        else:
            s = 2.0 * np.sqrt(1.0 + Rmat[2,2] - Rmat[0,0] - Rmat[1,1])
            w = (Rmat[1,0] - Rmat[0,1]) / s; x = (Rmat[0,2] + Rmat[2,0]) / s
            y = (Rmat[1,2] + Rmat[2,1]) / s; z = 0.25 * s
    return np.array((x, y, z, w), dtype=Rmat.dtype)

@njit(cache=True, fastmath=True)
def _quat2mat_single(q):
    x, y, z, w = q
    xx, yy, zz = x*x, y*y, z*z
    xy, xz, yz = x*y, x*z, y*z
    wx, wy, wz = w*x, w*y, w*z
    M = np.empty((3, 3), dtype=q.dtype)
    M[0,0]=1.0-2.0*(yy+zz); M[0,1]=2.0*(xy-wz); M[0,2]=2.0*(xz+wy)
    M[1,0]=2.0*(xy+wz); M[1,1]=1.0-2.0*(xx+zz); M[1,2]=2.0*(yz-wx)
    M[2,0]=2.0*(xz-wy); M[2,1]=2.0*(yz+wx); M[2,2]=1.0-2.0*(xx+yy)
    return M

@njit(cache=True, fastmath=True)
def _mat2quat_kernel(poses_mat, out):
    N = poses_mat.shape[0]
    for i in range(N):
        out[i, :3] = poses_mat[i, :3, 3]
        out[i, 3:] = _mat2quat_single(poses_mat[i, :3, :3])

@njit(cache=True, fastmath=True)
def _quat2mat_kernel(poses_quat, out):
    N = poses_quat.shape[0]
    for i in range(N):
        out[i, :3, 3] = poses_quat[i, :3]
        Rm = _quat2mat_single(poses_quat[i, 3:])
        out[i, 0, :3] = Rm[0]; out[i, 1, :3] = Rm[1]; out[i, 2, :3] = Rm[2]

def convert_pose_mat2quat(poses_mat):
    was_single = poses_mat.ndim == 2
    if was_single: poses_mat = poses_mat[None]
    out = np.empty((poses_mat.shape[0], 7), dtype=poses_mat.dtype)
    _mat2quat_kernel(poses_mat, out)
    return out[0] if was_single else out

def convert_pose_quat2mat(poses_quat):
    was_single = poses_quat.ndim == 1
    if was_single: poses_quat = poses_quat[None]
    out = np.empty((poses_quat.shape[0], 4, 4), dtype=poses_quat.dtype)
    out[:, 3, :] = np.array((0., 0., 0., 1.), dtype=poses_quat.dtype)
    _quat2mat_kernel(poses_quat, out)
    return out[0] if was_single else out

def mat2quat(rmat): return R.from_matrix(rmat).as_quat()
def quat2mat(quaternion): return R.from_quat(quaternion).as_matrix()
def euler2mat(euler): return R.from_euler("xyz", np.asarray(euler, dtype=np.float64)).as_matrix()
def mat2euler(rmat): return R.from_matrix(np.array(rmat)[:3, :3]).as_euler("xyz")
def euler2quat(euler): return R.from_euler("xyz", euler).as_quat()
def quat2euler(quat): return R.from_quat(quat).as_euler("xyz")
def quat2axisangle(quat): return R.from_quat(quat).as_rotvec()
def axisangle2quat(vec): return R.from_rotvec(vec).as_quat()

def mat2pose(hmat):
    return hmat[:3, 3], mat2quat(hmat[:3, :3])

def pose2mat(pose):
    homo = np.zeros((4, 4), dtype=pose[0].dtype)
    homo[:3, :3] = quat2mat(pose[1]); homo[:3, 3] = np.array(pose[0]); homo[3, 3] = 1.0
    return homo

def pose_inv(pose_mat):
    out = np.zeros((4, 4))
    out[:3, :3] = pose_mat[:3, :3].T
    out[:3, 3] = -out[:3, :3].dot(pose_mat[:3, 3])
    out[3, 3] = 1.0
    return out

def unit_vector(data, axis=None, out=None):
    data = np.array(data, dtype=np.float64, copy=True)
    if out is not None:
        out[:] = np.array(data)
        data = out
    length = np.dot(data, data) if axis is None else np.atleast_1d(np.sum(data*data, axis))
    np.sqrt(length, length)
    if axis is not None:
        length = np.expand_dims(length, axis)
    data /= length
    if out is None:
        return data

def quat_multiply(quaternion1, quaternion0):
    x0,y0,z0,w0 = quaternion0; x1,y1,z1,w1 = quaternion1
    return np.array((
        x1*w0+y1*z0-z1*y0+w1*x0, -x1*z0+y1*w0+z1*x0+w1*y0,
        x1*y0-y1*x0+z1*w0+w1*z0, -x1*x0-y1*y0-z1*z0+w1*w0,
    ), dtype=quaternion0.dtype)

def quat_conjugate(q): return np.array((-q[0], -q[1], -q[2], q[3]), dtype=q.dtype)
def quat_inverse(q): return quat_conjugate(q) / np.dot(q, q)
def quat_distance(q1, q0): return quat_multiply(q1, quat_inverse(q0))

def convert_pose_euler2mat(poses_euler):
    """Convert (N,6) [x,y,z,rx,ry,rz] to (N,4,4) matrices."""
    was_single = poses_euler.ndim == 1
    if was_single: poses_euler = poses_euler[None]
    N = poses_euler.shape[0]
    out = np.eye(4)[None].repeat(N, axis=0)
    out[:, :3, 3] = poses_euler[:, :3]
    for i in range(N):
        out[i, :3, :3] = euler2mat(poses_euler[i, 3:])
    return out[0] if was_single else out

def convert_pose_euler2quat(poses_euler):
    """Convert (N,6) [x,y,z,rx,ry,rz] to (N,7) [x,y,z,qx,qy,qz,qw]."""
    mats = convert_pose_euler2mat(poses_euler)
    return convert_pose_mat2quat(mats)

print(f"✅ transform_utils defined (numba available: {_TU_NUMBA_AVAILABLE})")

---
## Step 4 — Define: `droid_utils` (DROID dataset loading)

In [ ]:
# ============================================================
# droid_utils — DROID dataset loading utilities
# ============================================================
import os, json, glob, tempfile, shutil
import numpy as np
import cv2
from tqdm import tqdm
import h5py

try:
    import pyzed.sl as sl
except ImportError as exc:
    sl = None
    _PYZED_IMPORT_ERROR = exc
else:
    _PYZED_IMPORT_ERROR = None

def _require_pyzed():
    if sl is None:
        raise ImportError("pyzed is required for ZED SVO processing") from _PYZED_IMPORT_ERROR

def _binary_search_latest_range(arr, left, right, target):
    if arr[right] <= target or right == left:
        return arr[right]
    mid = ((left + right) >> 1) + 1
    if arr[mid] <= target:
        return _binary_search_latest_range(arr, mid, right, target)
    return _binary_search_latest_range(arr, left, mid - 1, target)

def _binary_search_latest(arr, target):
    if len(arr) <= 0:
        raise ValueError("input array should contain at least one element")
    return _binary_search_latest_range(arr, 0, len(arr) - 1, target)

def _binary_search_closest(arr, target):
    if target in arr:
        return target
    prev_idx = arr.index(_binary_search_latest(arr, target))
    if prev_idx == len(arr) - 1:
        return arr[prev_idx]
    prev_val = arr[prev_idx]; next_val = arr[prev_idx + 1]
    return prev_val if abs(prev_val - target) < abs(next_val - target) else next_val

def _resolve_svo_path(scene_path, svo_path):
    if not svo_path.startswith('/') and not is_gcs_path(svo_path):
        return os.path.join(scene_path, *svo_path.split('/')[-3:])
    return svo_path

def _init_camera_entries(metadata, include_wrist_cam):
    camera_names = ['wrist', 'ext1', 'ext2'] if include_wrist_cam else ['ext1', 'ext2']
    data_dict = {}; camera_specs = []
    for camera_name in camera_names:
        serial_key = f'{camera_name}_cam_serial'
        if serial_key not in metadata:
            raise KeyError(f"Camera {camera_name} not found in metadata")
        camera_serial = metadata[serial_key]
        data_dict[camera_serial] = {}
        if camera_name == 'wrist':
            data_dict[camera_serial]['extrinsic'] = np.eye(4)
        else:
            extrinsic_key = f'{camera_name}_cam_extrinsics'
            if extrinsic_key not in metadata:
                raise KeyError(f"Extrinsics for camera {camera_name} not found in metadata")
            extrinsic_6 = np.array(metadata[extrinsic_key])
            extrinsic_4x4 = convert_pose_euler2mat(extrinsic_6[None])[0]
            extrinsic_4x4 = np.linalg.inv(extrinsic_4x4)
            data_dict[camera_serial]['extrinsic'] = extrinsic_4x4
        camera_specs.append((camera_name, camera_serial))
    return data_dict, camera_specs

def _open_svo_camera(local_svo_path, include_depth):
    _require_pyzed()
    init_params = sl.InitParameters()
    init_params.set_from_svo_file(local_svo_path)
    init_params.svo_real_time_mode = False
    init_params.depth_mode = sl.DEPTH_MODE.ULTRA if include_depth else sl.DEPTH_MODE.NONE
    init_params.coordinate_units = sl.UNIT.METER
    zed = sl.Camera()
    err = zed.open(init_params)
    if err != sl.ERROR_CODE.SUCCESS:
        raise ValueError(f"Error opening SVO file {local_svo_path}: {err}")
    return zed, zed.get_camera_information()

def _extract_intrinsics_and_baseline(calibration_params, downscale_ratio):
    fx = calibration_params.left_cam.fx; fy = calibration_params.left_cam.fy
    cx = calibration_params.left_cam.cx; cy = calibration_params.left_cam.cy
    intrinsic = np.array([[fx, 0, cx], [0, fy, cy], [0, 0, 1]])
    baseline = calibration_params.stereo_transform.get_translation().get()[0]
    if downscale_ratio != 1.0:
        intrinsic[0, 0] *= downscale_ratio; intrinsic[1, 1] *= downscale_ratio
        intrinsic[0, 2] *= downscale_ratio; intrinsic[1, 2] *= downscale_ratio
    return intrinsic, baseline

def _extract_svo_frames(zed, camera_name, include_stereo, include_depth, downscale_ratio, max_frames):
    rgb_frames = []; right_frames = [] if include_stereo else None
    depth_frames = [] if include_depth else None
    left_image = sl.Mat()
    right_image = sl.Mat() if include_stereo else None
    depth_image = sl.Mat() if include_depth else None
    runtime_params = sl.RuntimeParameters()
    nb_frames = zed.get_svo_number_of_frames()
    if max_frames > 0: nb_frames = min(nb_frames, max_frames)
    timestamps = []; frame_count = 0
    with tqdm(total=nb_frames, desc=f"Processing {camera_name} camera") as pbar:
        while frame_count < nb_frames:
            err = zed.grab(runtime_params)
            if err == sl.ERROR_CODE.SUCCESS:
                zed.retrieve_image(left_image, sl.VIEW.LEFT)
                rgb = left_image.get_data().copy()
                if rgb.shape[2] == 4: rgb = cv2.cvtColor(rgb, cv2.COLOR_BGRA2RGB)
                if downscale_ratio != 1.0: rgb = cv2.resize(rgb, None, fx=downscale_ratio, fy=downscale_ratio, interpolation=cv2.INTER_LINEAR)
                rgb_frames.append(rgb)
                ts = zed.get_timestamp(sl.TIME_REFERENCE.IMAGE).get_milliseconds()
                timestamps.append(ts)
                if include_stereo:
                    zed.retrieve_image(right_image, sl.VIEW.RIGHT)
                    right = right_image.get_data().copy()
                    if right.shape[2] == 4: right = cv2.cvtColor(right, cv2.COLOR_BGRA2RGB)
                    if downscale_ratio != 1.0: right = cv2.resize(right, None, fx=downscale_ratio, fy=downscale_ratio, interpolation=cv2.INTER_LINEAR)
                    right_frames.append(right)
                if include_depth:
                    zed.retrieve_measure(depth_image, sl.MEASURE.DEPTH)
                    depth = depth_image.get_data().copy()
                    if downscale_ratio != 1.0: depth = cv2.resize(depth, None, fx=downscale_ratio, fy=downscale_ratio, interpolation=cv2.INTER_NEAREST)
                    depth_frames.append(depth)
                frame_count += 1; pbar.update(1)
            elif err == sl.ERROR_CODE.END_OF_SVOFILE_REACHED:
                print(f"End of SVO file reached for camera {camera_name}"); break
            else:
                raise ValueError(f"Error grabbing frame from SVO for camera {camera_name}: {err}")
    if not rgb_frames: raise ValueError(f"No frames extracted for camera {camera_name}")
    payload = {'rgb': np.stack(rgb_frames, axis=0), 'timestamps': np.array(timestamps)}
    if include_stereo: payload['right_frames'] = np.stack(right_frames, axis=0)
    if include_depth:
        payload['depth'] = np.stack(depth_frames, axis=0)
        payload['depth'] = np.nan_to_num(payload['depth'], nan=0.0, posinf=0.0, neginf=0.0, copy=False)
    return payload

def filter_by_timestamps(data, canonical_timestamps, scene_path, is_proprio=False, verbose=False):
    """Filter data to match canonical timestamps using binary search."""
    filtered_data = {}
    if is_proprio:
        temp_dir = None
        try:
            if is_gcs_path(scene_path): temp_dir = tempfile.mkdtemp()
            trajectory_path = os.path.join(scene_path, "trajectory.h5")
            local_trajectory_path = get_local_path(trajectory_path, temp_dir)
            with h5py.File(local_trajectory_path, 'r') as f:
                proprio_timestamps = np.array(f['observation']['timestamp']['robot_state']['read_end'])
        finally:
            if temp_dir and os.path.exists(temp_dir): shutil.rmtree(temp_dir)
        proprio_timestamps_list = proprio_timestamps.tolist()
        modality_errors = {}
        for modality in data:
            if modality in ['pre_sampled_points']:
                filtered_data[modality] = data[modality]
            else:
                assert len(data[modality]) == len(proprio_timestamps_list)
                aligned_data = []; errors = []
                for target_ts in canonical_timestamps:
                    closest_ts = _binary_search_closest(proprio_timestamps_list, target_ts)
                    idx = proprio_timestamps_list.index(closest_ts)
                    aligned_data.append(data[modality][idx])
                    if verbose: errors.append(abs(target_ts - closest_ts))
                filtered_data[modality] = np.array(aligned_data)
                if verbose and errors: modality_errors[modality] = np.array(errors) / 1000.0
    else:
        nontemporal_keys = ['intrinsic', 'extrinsic', 'baseline']
        for camera_serial in data:
            filtered_data[camera_serial] = {}
            for key in nontemporal_keys:
                if key in data[camera_serial]:
                    filtered_data[camera_serial][key] = data[camera_serial][key]
            camera_timestamps_list = data[camera_serial]['timestamps'].tolist()
            for key in data[camera_serial]:
                if key in nontemporal_keys: continue
                assert len(data[camera_serial][key]) == len(camera_timestamps_list)
                aligned_data = []
                for target_ts in canonical_timestamps:
                    closest_ts = _binary_search_closest(camera_timestamps_list, target_ts)
                    idx = camera_timestamps_list.index(closest_ts)
                    aligned_data.append(data[camera_serial][key][idx])
                filtered_data[camera_serial][key] = np.array(aligned_data)
    return filtered_data

def get_uuid(scene_path):
    """Extract UUID from metadata filename."""
    if is_gcs_path(scene_path):
        metadata_files = list_gcs_files(scene_path, "metadata_*.json")
        if not metadata_files: raise FileNotFoundError(f"No metadata files found in {scene_path}")
        return os.path.basename(metadata_files[0])[9:-5]
    metadata_files = glob.glob(os.path.join(scene_path, "metadata_*.json"))
    if not metadata_files: raise FileNotFoundError(f"No metadata files found in {scene_path}")
    return os.path.basename(metadata_files[0])[9:-5]

def get_metadata(scene_path):
    temp_dir = None
    try:
        if is_gcs_path(scene_path):
            temp_dir = tempfile.mkdtemp()
            metadata_files = list_gcs_files(scene_path, "metadata_*.json")
            if not metadata_files: raise FileNotFoundError(f"No metadata files found in {scene_path}")
            local_metadata_path = get_local_path(metadata_files[0], temp_dir)
            with open(local_metadata_path, 'r') as f: return json.load(f)
        else:
            metadata_files = glob.glob(os.path.join(scene_path, "metadata_*.json"))
            if not metadata_files: raise FileNotFoundError(f"No metadata files found in {scene_path}")
            with open(metadata_files[0], 'r') as f: return json.load(f)
    finally:
        if temp_dir and os.path.exists(temp_dir): shutil.rmtree(temp_dir)

def gather_data_dict(scene_path, downscale_ratio=1.0, include_stereo=False, include_depth=True, include_wrist_cam=False, max_frames=-1):
    """Gather data from DROID dataset SVO files."""
    _require_pyzed()
    data_dict = {}; temp_dir = None
    try:
        if is_gcs_path(scene_path): temp_dir = tempfile.mkdtemp()
        metadata = get_metadata(scene_path)
        data_dict, camera_specs = _init_camera_entries(metadata, include_wrist_cam)
        for camera_name, camera_serial in camera_specs:
            if f'{camera_name}_svo_path' not in metadata:
                raise KeyError(f"SVO path for camera {camera_name} not found in metadata")
            svo_path = metadata[f'{camera_name}_svo_path']
            resolved_svo_path = _resolve_svo_path(scene_path, svo_path)
            local_svo_path = get_local_path(resolved_svo_path, temp_dir)
            print(f"Processing SVO file for camera {camera_name}: {local_svo_path}")
            zed, camera_info = _open_svo_camera(local_svo_path, include_depth)
            try:
                calibration_params = camera_info.camera_configuration.calibration_parameters
                intrinsic, baseline = _extract_intrinsics_and_baseline(calibration_params, downscale_ratio)
                data_dict[camera_serial]['intrinsic'] = intrinsic
                if include_stereo: data_dict[camera_serial]['baseline'] = baseline
                frames_payload = _extract_svo_frames(zed, camera_name, include_stereo=include_stereo, include_depth=include_depth, downscale_ratio=downscale_ratio, max_frames=max_frames)
                data_dict[camera_serial].update(frames_payload)
            finally:
                zed.close()
            print(f"Extracted {data_dict[camera_serial]['rgb'].shape[0]} frames for camera {camera_serial}")
    finally:
        if temp_dir and os.path.exists(temp_dir): shutil.rmtree(temp_dir)
    return data_dict

def gather_trajectory(scene_path, max_frames=-1):
    """Gather proprioception trajectory from DROID dataset."""
    temp_dir = None
    try:
        if is_gcs_path(scene_path): temp_dir = tempfile.mkdtemp()
        trajectory_path = os.path.join(scene_path, "trajectory.h5")
        local_trajectory_path = get_local_path(trajectory_path, temp_dir)
        proprio_dict = {}
        with h5py.File(local_trajectory_path, 'r') as f:
            joint_positions = np.array(f['observation']['robot_state']['joint_positions'])
            joint_velocities = np.array(f['observation']['robot_state']['joint_velocities'])
            joint_torques = np.array(f['observation']['robot_state']['joint_torques_computed'])
            gripper_positions = np.array(f['observation']['robot_state']['gripper_position'])
            gripper_pose_6 = np.array(f['observation']['robot_state']['cartesian_position'])
            gripper_pose_7 = convert_pose_euler2quat(gripper_pose_6)
            proprio_timestamps = np.array(f['observation']['timestamp']['robot_state']['read_end'])
            if max_frames > 0:
                joint_positions = joint_positions[:max_frames]
                joint_velocities = joint_velocities[:max_frames]
                joint_torques = joint_torques[:max_frames]
                gripper_positions = gripper_positions[:max_frames]
                gripper_pose_7 = gripper_pose_7[:max_frames]
                proprio_timestamps = proprio_timestamps[:max_frames]
            proprio_dict['joint_positions'] = joint_positions
            proprio_dict['joint_velocities'] = joint_velocities
            proprio_dict['joint_torques'] = joint_torques
            proprio_dict['gripper_positions'] = gripper_positions * 0.725
            proprio_dict['gripper_pose'] = gripper_pose_7
            proprio_dict['timestamps'] = proprio_timestamps
    finally:
        if temp_dir and os.path.exists(temp_dir): shutil.rmtree(temp_dir)
    return proprio_dict

def load_robot_transforms(transforms_file):
    """Load gripper-to-wrist transforms from JSON file (dict of serial -> 4x4 matrix)."""
    if not os.path.exists(transforms_file):
        raise FileNotFoundError(f"Transform file not found: {transforms_file}")
    with open(transforms_file, 'r') as f:
        transforms_data = json.load(f)
    robot_transforms = {serial: np.array(data['mean_mat']) for serial, data in transforms_data.items()}
    print(f"Loaded gripper2wrist transforms for {len(robot_transforms)} robots.")
    return robot_transforms

def get_robot_serial(scene_path): return get_metadata(scene_path)['robot_serial']

print("✅ droid_utils defined (with gather_trajectory, filter_by_timestamps, load_robot_transforms)")

---
## Step 5 — Define: `DepthEstimator` (FoundationStereo wrapper)

In [ ]:
# ============================================================
# DepthEstimator — FoundationStereo stereo depth estimation
# ============================================================
import os, glob, sys
import numpy as np
import torch
from omegaconf import OmegaConf
from tqdm import tqdm

class DepthEstimator:
    """Standalone depth estimator using FoundationStereo model."""

    def __init__(self, ckpt_path, baseline=None, intrinsic=None, device='cuda', cfg_path=None):
        self.ckpt_path = ckpt_path
        self.baseline = baseline; self.intrinsic = intrinsic
        self.device = device; self.cfg_path = cfg_path
        self.model = None; self.is_loaded = False
        self.use_hierarchical = False; self.iters = 32
        self.load_model()
        print(f"DepthEstimator initialized with checkpoint: {ckpt_path}")

    def _resolve_ckpt_and_cfg(self):
        if self.cfg_path is not None:
            if not os.path.exists(self.cfg_path):
                raise FileNotFoundError(f"FoundationStereo config not found: {self.cfg_path}")
            if not os.path.isfile(self.ckpt_path):
                raise FileNotFoundError(f"FoundationStereo checkpoint not found: {self.ckpt_path}")
            return self.ckpt_path, self.cfg_path
        path = self.ckpt_path
        if os.path.isdir(path):
            matches = []
            for name in os.listdir(path):
                d = os.path.join(path, name)
                if not os.path.isdir(d): continue
                cfg = os.path.join(d, "cfg.yaml")
                pths = sorted(glob.glob(os.path.join(d, "*.pth")))
                if os.path.exists(cfg) and pths:
                    best = [p for p in pths if os.path.basename(p).startswith("model_best")]
                    matches.append((best[0] if best else pths[0], cfg))
            assert len(matches) == 1, f"Expected 1 subdir with cfg.yaml+.pth, found {len(matches)}"
            return matches[0]
        else:
            assert os.path.isfile(path), f"Checkpoint not found: {path}"
            cfg = os.path.join(os.path.dirname(path), "cfg.yaml")
            assert os.path.exists(cfg), f"Config not found: {cfg}"
            return path, cfg

    def load_model(self):
        if self.is_loaded: return
        repo_root = REPO_ROOT  # set in Step 0
        fs_root = os.path.join(repo_root, "third_party", "FoundationStereo")
        if fs_root not in sys.path: sys.path.append(fs_root)
        from core.utils.utils import InputPadder
        from core.foundation_stereo import FoundationStereo
        self.InputPadder = InputPadder
        self.ckpt_path, cfg_path = self._resolve_ckpt_and_cfg()
        cfg = OmegaConf.load(cfg_path)
        if "vit_size" not in cfg:
            raise ValueError(f"FoundationStereo cfg missing vit_size: {cfg_path}")
        print(f"Loading FoundationStereo model from {self.ckpt_path}")
        import time
        start_time = time.time()
        while time.time() - start_time < 600:
            try:
                self.model = FoundationStereo(cfg); break
            except Exception as e:
                if any(k in str(e).lower() for k in ["rate limit", "connection without response", "too many requests"]):
                    print(f"Retrying due to: {e}"); time.sleep(5)
                else:
                    raise
        else:
            raise RuntimeError("FoundationStereo init failed after timeout")
        ckpt = torch.load(self.ckpt_path, map_location='cpu')
        self.model.load_state_dict(ckpt['model'])
        if self.device == 'cuda' and torch.cuda.is_available(): self.model.cuda()
        self.model.eval(); self.is_loaded = True
        print(f"FoundationStereo loaded (step: {ckpt.get('global_step','?')}, epoch: {ckpt.get('epoch','?')})")

    def set_camera_params(self, baseline, intrinsic):
        self.baseline = baseline; self.intrinsic = intrinsic
        print(f"Camera params set - baseline: {baseline:.4f}m, fx: {intrinsic[0,0]:.2f}")

    def _prepare_image(self, image):
        if len(image.shape) == 2: image = np.stack([image, image, image], axis=2)
        elif image.shape[2] == 1: image = np.repeat(image, 3, axis=2)
        if image.dtype != np.uint8:
            image = (image * 255).astype(np.uint8) if image.max() <= 1.0 else image.astype(np.uint8)
        return image

    def _disparity_to_depth(self, disp):
        h, w = disp.shape
        xx = torch.arange(w, device=disp.device).view(1, w).expand(h, w)
        us_right = xx - disp; invalid = us_right < 0
        disp_valid = disp.clone(); disp_valid[invalid] = float('inf')
        depth = torch.tensor(self.intrinsic[0,0], device=disp.device) * torch.tensor(self.baseline, device=disp.device) / disp_valid
        depth[torch.isinf(depth) | torch.isnan(depth)] = 0.0
        return depth.cpu().numpy()

    @torch.inference_mode()
    @torch.cuda.amp.autocast(True)
    def infer_depth(self, left_image, right_image):
        if not self.is_loaded: self.load_model()
        left_image = self._prepare_image(left_image); right_image = self._prepare_image(right_image)
        device = next(self.model.parameters()).device
        left_t = torch.as_tensor(left_image).to(device).float().permute(2,0,1).unsqueeze(0)
        right_t = torch.as_tensor(right_image).to(device).float().permute(2,0,1).unsqueeze(0)
        padder = self.InputPadder(left_t.shape, divis_by=32, force_square=False)
        left_t, right_t = padder.pad(left_t, right_t)
        disp = self.model.forward(left_t.contiguous(), right_t.contiguous(), iters=self.iters, test_mode=True)
        disp = padder.unpad(disp.float()).squeeze()
        assert self.baseline is not None and self.intrinsic is not None
        return self._disparity_to_depth(disp)

    @torch.inference_mode()
    @torch.cuda.amp.autocast(True)
    def infer_depth_batch(self, left_images, right_images, batch_size=32):
        if not self.is_loaded: self.load_model()
        assert self.baseline is not None and self.intrinsic is not None
        num_frames = len(left_images)
        assert len(right_images) == num_frames
        depth_frames = []
        for i in tqdm(range(0, num_frames, batch_size), desc="Inferring depth"):
            batch_end = min(i + batch_size, num_frames)
            depth_np = self._infer_depth_batch_internal(left_images[i:batch_end], right_images[i:batch_end])
            for j in range(batch_end - i): depth_frames.append(depth_np[j])
        return np.stack(depth_frames, axis=0)

    @torch.inference_mode()
    @torch.cuda.amp.autocast(True)
    def _infer_depth_batch_internal(self, left_batch, right_batch):
        device = next(self.model.parameters()).device
        left_t = torch.from_numpy(np.stack(left_batch).astype(np.float32)).to(device).permute(0,3,1,2).contiguous()
        right_t = torch.from_numpy(np.stack(right_batch).astype(np.float32)).to(device).permute(0,3,1,2).contiguous()
        B = left_t.shape[0]
        padder = self.InputPadder(left_t.shape, divis_by=32, force_square=False)
        left_t, right_t = padder.pad(left_t, right_t)
        disp = self.model.forward(left_t, right_t, iters=self.iters, test_mode=True)
        disp = padder.unpad(disp.float()).squeeze(1)  # [B, H, W]
        h, w = disp.shape[1:]
        xx = torch.arange(w, device=disp.device).view(1,1,w).expand(B,h,w)
        us_right = xx - disp; invalid = us_right < 0
        disp_valid = disp.clone(); disp_valid[invalid] = float('inf')
        depth = torch.tensor(self.intrinsic[0,0], device=disp.device) * torch.tensor(self.baseline, device=disp.device) / disp_valid
        depth[torch.isinf(depth) | torch.isnan(depth)] = 0.0
        return depth.cpu().numpy()

print("✅ DepthEstimator defined")

In [ ]:
import os
import numpy as np
import h5py
import random
from tqdm import tqdm

# FoundationStereo checkpoint paths (relative to REPO_ROOT set in Step 0)
DEPTH_ESTIMATOR_CKPT_PATH = os.path.join(
    REPO_ROOT, "checkpoints", "foundationstereo", "23-51-11", "model_best_bp2.pth"
)
DEPTH_ESTIMATOR_CFG_PATH = os.path.join(
    REPO_ROOT, "assets", "foundationstereo", "23-51-11", "cfg.yaml"
)

print("Depth estimator ckpt path:", DEPTH_ESTIMATOR_CKPT_PATH)
print("Depth estimator cfg path :", DEPTH_ESTIMATOR_CFG_PATH)

In [ ]:
def check_depth_file_exists(output_dir, uuid):
    """
    Check if depth file exists and has valid structure with write_complete flags.

    Args:
        output_dir (str): Output directory for depth files
        uuid (str): Episode UUID

    Returns:
        bool: True if valid depth file exists, False otherwise
    """
    h5_path = os.path.join(output_dir, "depth", f"{uuid}_depth.h5")

    if not os.path.exists(h5_path):
        return False

    with h5py.File(h5_path, 'r') as f:
        if 'metadata' not in f:
            raise ValueError(f"Depth file missing metadata group: {h5_path}")

        metadata = f['metadata']
        if 'write_complete' not in metadata.attrs or not metadata.attrs['write_complete']:
            raise ValueError(f"Depth file metadata incomplete: {h5_path}")

        camera_groups = [key for key in f.keys() if key != 'metadata']
        if not camera_groups:
            raise ValueError(f"Depth file missing camera groups: {h5_path}")

        for camera_group_name in camera_groups:
            camera_group = f[camera_group_name]
            required_keys = ['depth', 'timestamps']
            for key in required_keys:
                if key not in camera_group:
                    raise ValueError(f"Depth file missing {key} in {camera_group_name}: {h5_path}")
                if 'write_complete' not in camera_group[key].attrs or not camera_group[key].attrs['write_complete']:
                    raise ValueError(f"Depth file {key} incomplete in {camera_group_name}: {h5_path}")

    return True

print("✅ check_depth_file_exists defined")

In [ ]:
def compute_depth_for_scene(
    scene_path,
    output_dir,
    depth_estimator,
    downscale_ratio=1.0,
    batch_size=32,
):
    """
    Compute stereo depth for all cameras in a single scene.

    Args:
        scene_path (str): Path to the scene directory
        output_dir (str): Output directory for depth files
        depth_estimator (DepthEstimator): Initialized depth estimator
        downscale_ratio (float): Downscale ratio for the images
        batch_size (int): Batch size for depth estimation

    Returns:
        bool: True if successful
    """
    uuid = get_uuid(scene_path)

    if check_depth_file_exists(output_dir, uuid):
        print(f"[{get_time_str()}] Depth already computed for {uuid}, skipping")
        return True

    print(f"[{get_time_str()}] Processing scene: {uuid}")

    data_dict = gather_data_dict(
        scene_path,
        downscale_ratio=downscale_ratio,
        include_stereo=True,
        include_depth=False,
    )

    depth_dir = os.path.join(output_dir, "depth")
    h5_path = os.path.join(depth_dir, f"{uuid}_depth.h5")
    os.makedirs(depth_dir, exist_ok=True)

    with h5py.File(h5_path, 'w') as f:
        metadata_group = f.create_group('metadata')
        metadata_group.attrs['uuid'] = uuid
        metadata_group.attrs['camera_count'] = len(data_dict)

        total_cameras = len(data_dict)
        frame_count = None

        for i, (camera_serial, camera_data) in enumerate(data_dict.items()):
            print(f"[{get_time_str()}] Processing camera {i+1}/{total_cameras}: {camera_serial}")

            left_frames = camera_data['rgb']
            right_frames = camera_data['right_frames']
            timestamps = camera_data['timestamps']
            intrinsic = camera_data['intrinsic']
            baseline = camera_data['baseline']
            if frame_count is None:
                frame_count = len(left_frames)

            depth_estimator.set_camera_params(baseline, intrinsic)

            print(f"[{get_time_str()}] Computing depth for {len(left_frames)} frames...")
            depth_frames = depth_estimator.infer_depth_batch(left_frames, right_frames, batch_size=batch_size)

            depth_mm = (depth_frames * 1000.0).astype(np.float32)
            depth_mm = np.clip(depth_mm, 0, 65535)
            depth_uint16 = depth_mm.astype(np.uint16)

            camera_type = "ext"
            camera_group_name = f"{camera_serial}+{camera_type}"

            camera_group = f.create_group(camera_group_name)
            camera_group.attrs['camera_serial'] = camera_serial
            camera_group.attrs['camera_type'] = camera_type

            depth_dataset = camera_group.create_dataset(
                'depth', data=depth_uint16,
                compression='gzip', compression_opts=9, shuffle=True, chunks=True
            )
            depth_dataset.attrs['write_complete'] = True
            depth_dataset.attrs['units'] = 'millimeters'
            depth_dataset.attrs['dtype'] = 'uint16'

            timestamps_dataset = camera_group.create_dataset('timestamps', data=timestamps)
            timestamps_dataset.attrs['write_complete'] = True
            timestamps_dataset.attrs['units'] = 'milliseconds'

            print(f"[{get_time_str()}] Saved depth data for camera {camera_group_name}: {depth_uint16.shape}")

        metadata_group.attrs['frame_count'] = frame_count
        metadata_group.attrs['write_complete'] = True

    print(f"[{get_time_str()}] Successfully saved depth data to {h5_path}")
    return True

print("✅ compute_depth_for_scene defined")

### Compute Depth — Configuration

Edit the variables below, then run the next cells to execute.

In [ ]:
# ============================================================
# Configuration — edit these values
# ============================================================

# Use the episodes list from the cloned repo (droid_paths.txt has full GCS paths)
# For a quick test, use episodes_debug.txt (100 scenes) instead:
#   INPUT_FILE = os.path.join(REPO_ROOT, "real", "droid_paths_debug.txt")
INPUT_FILE        = os.path.join(REPO_ROOT, "real", "droid_paths.txt")

# Prefix for the scene paths (if droid_paths.txt has relative names like
# 'AUTOLab+0d4edc83+2023-10-21-19h-02m-09s', set this to prepend the GCS prefix;
# if paths already start with gs://, leave empty)
SCENE_PATH_PREFIX = "gs://gresearch/robotics/droid_raw/1.0.1/"  # <-- set to "" if paths are already absolute

OUTPUT_DIR        = "/content/droid_output"  # output directory for depth H5 files
RANK              = 0                        # process rank (0-indexed)
WORLD_SIZE        = 1                        # total number of parallel workers
DOWNSCALE_RATIO   = 0.5                      # image downscale ratio (raw is 1280x720)
BATCH_SIZE        = 12                       # batch size (>12 may cause cudnn error with AMP)
FOUNDATION_STEREO_CKPT = DEPTH_ESTIMATOR_CKPT_PATH
FOUNDATION_STEREO_CFG  = DEPTH_ESTIMATOR_CFG_PATH
ALLOW_GCS_STREAMING    = True                # True: stream from GCS (no local cache needed)

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"INPUT_FILE: {INPUT_FILE}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

In [ ]:
# Read and partition scene paths
with open(INPUT_FILE, 'r') as f:
    all_paths = [line.strip() for line in f if line.strip()]

# Prepend GCS prefix if paths are relative (don't start with gs:// or /)
if SCENE_PATH_PREFIX:
    all_paths = [
        (SCENE_PATH_PREFIX + p) if not p.startswith('gs://') and not p.startswith('/') else p
        for p in all_paths
    ]
    print(f"Prepended prefix '{SCENE_PATH_PREFIX}' to relative paths")
    print(f"Example path: {all_paths[0]}")

enforce_gcs_cache_policy(
    all_paths,
    stage_name="compute_depth",
    require_cache=True,
    allow_streaming=ALLOW_GCS_STREAMING,
)

# Shuffle with fixed seed for reproducibility and load balancing
random.seed(42)
random.shuffle(all_paths)

# Validate distributed args
if WORLD_SIZE <= 0:
    raise ValueError(f"WORLD_SIZE must be >= 1, got {WORLD_SIZE}")
if not (0 <= RANK < WORLD_SIZE):
    raise ValueError(f"RANK must be in [0, WORLD_SIZE), got rank={RANK}, world_size={WORLD_SIZE}")

paths_to_process = all_paths[RANK::WORLD_SIZE]
print(f"[{get_time_str()}] Rank {RANK}/{WORLD_SIZE}: Processing {len(paths_to_process)}/{len(all_paths)} scenes")

In [ ]:
# Initialize DepthEstimator (defined above in Step 5)
print(f"[{get_time_str()}] Initializing depth estimator...")
if not os.path.exists(FOUNDATION_STEREO_CKPT):
    raise FileNotFoundError(f"FoundationStereo checkpoint not found: {FOUNDATION_STEREO_CKPT}")
if not os.path.exists(FOUNDATION_STEREO_CFG):
    raise FileNotFoundError(f"FoundationStereo cfg not found: {FOUNDATION_STEREO_CFG}")

depth_estimator = DepthEstimator(
    FOUNDATION_STEREO_CKPT,
    device='cuda',
    cfg_path=FOUNDATION_STEREO_CFG,
)
print(f"[{get_time_str()}] Depth estimator ready.")

In [ ]:
# Process all scenes
total_processed = 0
total_successful = 0

for scene_path in tqdm(paths_to_process, desc="Processing scenes"):
    total_processed += 1
    success = compute_depth_for_scene(
        scene_path,
        OUTPUT_DIR,
        depth_estimator,
        downscale_ratio=DOWNSCALE_RATIO,
        batch_size=BATCH_SIZE,
    )
    if success:
        total_successful += 1

print(f"\n[{get_time_str()}] Processing completed!")
print(f"  Total scenes processed: {total_processed}")
print(f"  Successful: {total_successful}")
print(f"  Success rate: {100*total_successful/total_processed:.1f}%")

---
## Part 2: Compute Extrinsics

Extrinsics optimization using VGGT initialization and robot mesh rendering.
Takes estimated extrinsics (from VGGT) and optimizes them using precomputed stereo depth (from FoundationStereo).

---
## Step 6 — Define: `extrinsics_io`, `vggt_forward`, `compute_extrinsics_utils`

In [ ]:
# ============================================================
# extrinsics_io — I/O helpers for extrinsics pipeline outputs
# ============================================================
from __future__ import annotations
import json, os, time, h5py
from typing import Callable
import numpy as np

def load_precomputed_depth(output_dir, uuid, data_dict, wrist_serial, log_fn):
    """Load precomputed FoundationStereo depth from H5 into data_dict."""
    start = time.time()
    h5_path = os.path.join(output_dir, "depth", f"{uuid}_depth.h5")
    if not os.path.exists(h5_path):
        raise FileNotFoundError(f"Precomputed depth file not found: {h5_path}")
    with h5py.File(h5_path, "r") as f:
        if "metadata" not in f:
            raise ValueError(f"Invalid depth file: missing metadata in {h5_path}")
        metadata = f["metadata"]
        if "write_complete" not in metadata.attrs:
            raise ValueError(f"Depth file missing write_complete flag: {h5_path}")
        if not metadata.attrs["write_complete"]:
            raise ValueError(f"Depth file not complete: {h5_path}")
        camera_groups = [key for key in f.keys() if key != "metadata"]
        for camera_group_name in camera_groups:
            camera_serial, _camera_type = camera_group_name.split("+")
            if camera_serial not in data_dict:
                raise ValueError(f"Camera {camera_serial} in depth file but not in data_dict")
            camera_group = f[camera_group_name]
            depth_uint16 = camera_group["depth"][:]  # [T, H, W]
            depth_timestamps = camera_group["timestamps"][:]  # [T]
            depth_meters = depth_uint16.astype(np.float32) / 1000.0
            canonical_timestamps = data_dict[camera_serial]["timestamps"]
            aligned_depth_frames = []
            for canonical_ts in canonical_timestamps:
                time_diffs = np.abs(depth_timestamps - canonical_ts)
                closest_idx = np.argmin(time_diffs)
                if time_diffs[closest_idx] > 50:
                    log_fn(f"Warning: Large ts diff ({time_diffs[closest_idx]:.1f}ms) for camera {camera_serial}")
                aligned_depth_frames.append(depth_meters[closest_idx])
            aligned_depth = np.stack(aligned_depth_frames, axis=0)
            data_dict[camera_serial]["stereo_depth"] = aligned_depth
            data_dict[camera_serial]["stereo_intrinsics"] = data_dict[camera_serial]["measured_intrinsics"]
            log_fn(f"Loaded depth for camera {camera_serial}: {aligned_depth.shape}, range: {aligned_depth.min():.3f}-{aligned_depth.max():.3f}m")
    num_cams = len([k for k in data_dict.keys() if k != wrist_serial])
    log_fn(f"Successfully loaded precomputed depth for {num_cams} cameras (time taken: {time.time() - start:.2f}s)")

def write_camera_results(output_dir, uuid, scene_path, data_dict, wrist_serial, optimization_metrics, error_info, log_fn):
    """Write camera JSON results to output_dir/cameras."""
    cameras_dir = os.path.join(output_dir, "cameras")
    os.makedirs(cameras_dir, exist_ok=True)
    results = {"uuid": uuid, "scene_path": scene_path}
    if error_info is not None:
        results["error_info"] = error_info
    if optimization_metrics:
        results["optimization_summary"] = optimization_metrics
    for camera_serial in data_dict:
        if camera_serial == wrist_serial:
            continue
        camera_data = data_dict[camera_serial]
        camera_result = {}
        if "vggt_extrinsics" in camera_data:
            camera_result["vggt_extrinsics"] = camera_data["vggt_extrinsics"].tolist()
        if error_info is None and "optimized_extrinsics" in camera_data:
            camera_result["optimized_extrinsics"] = camera_data["optimized_extrinsics"].tolist()
        if "measured_intrinsics" in camera_data:
            camera_result["measured_intrinsics"] = camera_data["measured_intrinsics"].tolist()
        if "vggt_intrinsics" in camera_data:
            camera_result["vggt_intrinsics"] = camera_data["vggt_intrinsics"].tolist()
        results[camera_serial] = camera_result
    json_path = os.path.join(cameras_dir, f"{uuid}_cameras.json")
    with open(json_path, "w") as f:
        json.dump(results, f, indent=2)
    log_fn(f"Camera results saved to: {json_path}")
    return json_path

print("✅ extrinsics_io defined")

In [ ]:
# ============================================================
# vggt_forward — VGGT forward-pass wrapper
# ============================================================
import os, sys, shutil, tempfile
from contextlib import contextmanager
import numpy as np
import torch
from PIL import Image

# Add vggt to sys.path
_VGGT_ROOT = os.path.join(REPO_ROOT, "third_party", "vggt")
if _VGGT_ROOT not in sys.path:
    sys.path.append(_VGGT_ROOT)

try:
    from vggt.utils.pose_enc import pose_encoding_to_extri_intri
    from vggt.utils.load_fn import load_and_preprocess_images
except ImportError as e:
    print(f"Warning: Could not import VGGT utilities: {e}")
    print("Ensure the vggt submodule is initialized inside the cloned PointWorld repo.")
    raise

@contextmanager
def stage_vggt_images(images):
    """Write RGB images to a temp dir for VGGT and yield file paths."""
    temp_dir = tempfile.mkdtemp(prefix="vggt_frames_")
    paths = []
    try:
        for idx, img in enumerate(images):
            path = os.path.join(temp_dir, f"frame_{idx:05d}.png")
            Image.fromarray(img).save(path)
            paths.append(path)
        yield paths
    finally:
        shutil.rmtree(temp_dir, ignore_errors=True)

class VGGTForwardPass:
    """VGGT forward pass for camera extrinsics/intrinsics."""
    def __init__(self, model, device="cuda"):
        self.model = model; self.device = device

    @torch.inference_mode()
    @torch.cuda.amp.autocast(enabled=True)
    def __call__(self, image_paths):
        images_tensor = load_and_preprocess_images(image_paths).to(self.device)
        images_tensor = images_tensor[None]  # [batch=1, N, 3, H, W]
        aggregated_tokens_list, _ = self.model.aggregator(images_tensor)
        pose_enc = self.model.camera_head(aggregated_tokens_list)[-1]
        extrinsic_np, intrinsic_np = pose_encoding_to_extri_intri(pose_enc, images_tensor.shape[-2:])
        extrinsic_np = extrinsic_np[0].cpu().numpy()  # [N, 3, 4]
        intrinsic_np = intrinsic_np[0].cpu().numpy()  # [N, 3, 3]
        extrinsic_4x4_list = []
        for i in range(extrinsic_np.shape[0]):
            M = np.eye(4, dtype=np.float32); M[:3, :4] = extrinsic_np[i]
            extrinsic_4x4_list.append(M)
        return {
            "extrinsics_3x4": extrinsic_np,
            "extrinsics_4x4": np.stack(extrinsic_4x4_list, axis=0),
            "intrinsics": intrinsic_np,
        }

print("✅ vggt_forward defined")

In [ ]:
# ============================================================
# compute_extrinsics_utils — robot renderer and optimization utils
# ============================================================
import os, json
import numpy as np
if not hasattr(np, "float"):
    np.float = float  # urdfpy compatibility alias
from typing import List
import torch
import urdfpy
import h5py

REAL_DIR = os.path.join(REPO_ROOT, "real")
DEFAULT_GRIPPER2WRIST_TRANSFORMS_PATH = os.path.join(
    REAL_DIR, "gripper2wrist_transforms.json"
)

class RobotVisibilityError(Exception):
    """Raised when robot is not visible in cameras during optimization."""
    def __init__(self, message, error_type="no_robot_visible"):
        self.error_type = error_type
        super().__init__(message)

def check_precomputed_depth_exists(scene_path, output_dir, uuid=None):
    if uuid is None: uuid = get_uuid(scene_path)
    h5_path = os.path.join(output_dir, "depth", f"{uuid}_depth.h5")
    if not os.path.exists(h5_path):
        return False, h5_path, f"Precomputed depth file not found: {h5_path}"
    with h5py.File(h5_path, 'r') as f:
        if 'metadata' not in f: raise ValueError(f"Invalid depth file: missing metadata in {h5_path}")
        metadata = f['metadata']
        if 'write_complete' not in metadata.attrs: raise ValueError(f"Depth file missing write_complete flag: {h5_path}")
        if not metadata.attrs['write_complete']: raise ValueError(f"Depth file not complete: {h5_path}")
        camera_groups = [key for key in f.keys() if key != 'metadata']
        if len(camera_groups) == 0: raise ValueError(f"No camera groups found in depth file: {h5_path}")
    return True, h5_path, None

def check_camera_results_exist(scene_path, output_dir, uuid=None):
    if uuid is None: uuid = get_uuid(scene_path)
    json_path = os.path.join(output_dir, "cameras", f"{uuid}_cameras.json")
    if not os.path.exists(json_path):
        return False, json_path, f"Camera results file not found: {json_path}"
    with open(json_path, 'r') as f:
        data = json.load(f)
    if 'uuid' not in data: raise ValueError(f"Invalid camera results file: missing uuid in {json_path}")
    if data['uuid'] != uuid: raise ValueError(f"UUID mismatch: expected {uuid}, got {data['uuid']}")
    camera_count = sum(1 for k, v in data.items() if k not in ['uuid', 'optimization_summary', 'error_info', 'scene_path'] and isinstance(v, dict))
    return True, json_path, None

def deduplicate_coordinates(xy: torch.Tensor, threshold_pixels: float = 0.5):
    """Torch-native coordinate deduplication using grid cells."""
    if xy.numel() == 0:
        return [] if xy.ndim == 3 else torch.empty(0, dtype=torch.long)
    q = torch.round(xy / threshold_pixels).to(torch.int32)
    stride = 131_071
    code = q[..., 0] * stride + q[..., 1]
    if xy.ndim == 2:
        idx = torch.arange(code.shape[0], device=xy.device)
        uniq, inv = torch.unique(code, return_inverse=True, sorted=False)
        first = torch.full_like(uniq, fill_value=code.shape[0], dtype=torch.long)
        first.scatter_reduce_(0, inv, idx, reduce="amin")
        return first
    else:
        B, N = code.shape[:2]
        batch_idx = torch.arange(N, device=xy.device).expand(B, N)
        out: List[torch.Tensor] = []
        for b in range(B):
            uniq, inv = torch.unique(code[b], return_inverse=True, sorted=False)
            first = torch.full_like(uniq, fill_value=N, dtype=torch.long)
            first.scatter_reduce_(0, inv, batch_idx[b], reduce="amin")
            out.append(first)
        return out

def sample_depth_with_grid_sample(depth, xy, align_corners=True):
    """Bilinear-interpolated depth lookup via F.grid_sample."""
    assert depth.device == xy.device
    assert depth.ndim in (2, 3)
    assert xy.ndim == depth.ndim
    if depth.ndim == 2:
        H, W = depth.shape
        x, y = xy[:, 0], xy[:, 1]
        grid = torch.stack((2*x/(W-1)-1, 2*y/(H-1)-1), dim=1).unsqueeze(0).unsqueeze(0)
        out = torch.nn.functional.grid_sample(depth.unsqueeze(0).unsqueeze(0), grid, mode="bilinear", padding_mode="zeros", align_corners=align_corners)
        return out.view(-1)
    else:
        B, H, W = depth.shape; B2, N = xy.shape[:2]
        x, y = xy[..., 0], xy[..., 1]
        grid = torch.stack((2*x/(W-1)-1, 2*y/(H-1)-1), dim=2).unsqueeze(1)
        out = torch.nn.functional.grid_sample(depth.unsqueeze(1), grid, mode="bilinear", padding_mode="zeros", align_corners=align_corners)
        return out.squeeze(1).squeeze(1)

def pose_6dof_to_matrix(pose_6dof):
    """Convert 6-DOF pose tensor to 4x4 transformation matrix."""
    x, y, z, roll, pitch, yaw = pose_6dof
    cos_r, sin_r = torch.cos(roll), torch.sin(roll)
    cos_p, sin_p = torch.cos(pitch), torch.sin(pitch)
    cos_y, sin_y = torch.cos(yaw), torch.sin(yaw)
    Rmat = torch.zeros(3, 3, device=pose_6dof.device, dtype=pose_6dof.dtype)
    Rmat[0,0]=cos_y*cos_p; Rmat[0,1]=cos_y*sin_p*sin_r-sin_y*cos_r; Rmat[0,2]=cos_y*sin_p*cos_r+sin_y*sin_r
    Rmat[1,0]=sin_y*cos_p; Rmat[1,1]=sin_y*sin_p*sin_r+cos_y*cos_r; Rmat[1,2]=sin_y*sin_p*cos_r-cos_y*sin_r
    Rmat[2,0]=-sin_p;      Rmat[2,1]=cos_p*sin_r;                    Rmat[2,2]=cos_p*cos_r
    T = torch.eye(4, device=pose_6dof.device, dtype=pose_6dof.dtype)
    T[:3,:3] = Rmat; T[:3,3] = torch.stack([x,y,z])
    return T

class RobotMeshRenderer:
    """Simplified robot mesh renderer for extrinsics optimization."""
    def __init__(self, urdf_path, device="cuda", total_samples=25000):
        self.device = device; self.urdf_path = urdf_path; self.dtype = torch.float32
        assert os.path.exists(urdf_path), f"URDF file not found: {urdf_path}"
        self.robot_urdf = urdfpy.URDF.load(urdf_path)
        print(f"Loaded URDF from: {urdf_path}")
        self.mesh_points = {}; self.total_samples = total_samples
        self.fk_cache = {}; self.world_points_cache = {}

    def _get_forward_kinematics(self, joint_positions, gripper_position):
        cache_key = tuple(np.round(joint_positions, 4).tolist() + [round(float(gripper_position), 4)])
        if cache_key in self.fk_cache: return self.fk_cache[cache_key]
        cfg = {'finger_joint': float(gripper_position)}
        for ji in range(7): cfg[f'panda_joint{ji+1}'] = float(joint_positions[ji])
        fk_result = self.robot_urdf.visual_trimesh_fk(cfg=cfg)
        self.fk_cache[cache_key] = fk_result
        return fk_result

    def _sample_mesh_points(self, fk_result):
        mesh_names, mesh_objects, mesh_areas = [], [], []
        for i, mesh in enumerate(fk_result):
            mesh_name = get_mesh_name(mesh, i)
            if mesh.area <= 0: continue
            effective_area = mesh.area
            if 'hand_camera_part' in mesh_name.lower(): effective_area *= 0.000001
            mesh_names.append(mesh_name); mesh_objects.append(mesh); mesh_areas.append(effective_area)
        if not mesh_names: raise ValueError("No meshes with positive area found.")
        total_area = sum(mesh_areas)
        if total_area <= 0: raise ValueError("Total mesh area is non-positive.")
        for name, mesh, area in zip(mesh_names, mesh_objects, mesh_areas):
            count = max(200, int(self.total_samples * area / total_area))
            points_3d = mesh.sample(count)
            self.mesh_points[name] = torch.from_numpy(points_3d.astype(np.float32)).to(device=self.device, dtype=self.dtype)

    def _get_world_points(self, fk_result):
        fk_poses_key = []
        for i, mesh in enumerate(fk_result):
            mesh_name = get_mesh_name(mesh, i)
            if mesh_name in self.mesh_points or not self.mesh_points:
                pose_flat = fk_result[mesh].flatten()
                fk_poses_key.extend(np.round(pose_flat, 4).tolist())
        cache_key = tuple(fk_poses_key)
        if cache_key in self.world_points_cache: return self.world_points_cache[cache_key]
        if not self.mesh_points: self._sample_mesh_points(fk_result)
        fk_poses = {}
        for i, mesh in enumerate(fk_result):
            mesh_name = get_mesh_name(mesh, i)
            if mesh_name in self.mesh_points: fk_poses[mesh_name] = fk_result[mesh]
        all_world_points = []
        for mesh_name, local_points in self.mesh_points.items():
            if mesh_name not in fk_poses: continue
            pose = torch.as_tensor(fk_poses[mesh_name], dtype=self.dtype, device=self.device)
            ones = torch.ones((local_points.shape[0], 1), device=self.device, dtype=self.dtype)
            pts_h = torch.cat([local_points, ones], dim=1)
            world_pts = torch.mm(pts_h, pose.T)[:, :3]
            all_world_points.append(world_pts)
        assert len(all_world_points) > 0, "No valid robot points found"
        result = torch.cat(all_world_points, dim=0)
        self.world_points_cache[cache_key] = result
        return result

    def resample(self, total_samples=None):
        if total_samples is not None and total_samples > 0: self.total_samples = total_samples
        self.mesh_points.clear(); self.world_points_cache.clear()

def load_robot_transforms(transforms_file):
    """Load gripper-to-wrist transforms from JSON file."""
    if not os.path.exists(transforms_file):
        raise FileNotFoundError(f"Transform file not found: {transforms_file}")
    with open(transforms_file, 'r') as f:
        transforms_data = json.load(f)
    robot_transforms = {serial: np.array(data['mean_mat']) for serial, data in transforms_data.items()}
    print(f"Loaded gripper2wrist transforms for {len(robot_transforms)} robots.")
    return robot_transforms

def get_robot_transform(robot_serial, transforms_file=DEFAULT_GRIPPER2WRIST_TRANSFORMS_PATH):
    """Get gripper-to-wrist transform for a robot."""
    all_transforms = load_robot_transforms(transforms_file)
    if robot_serial not in all_transforms:
        available = list(all_transforms.keys())
        raise ValueError(f"Robot serial {robot_serial} not found. Available: {available[:10]}")
    return np.array(all_transforms[robot_serial])

print("✅ compute_extrinsics_utils defined")

---
## Step 7 — Define: `ExtrinsicsOptimizer` and `process_single_scene`

In [ ]:
# ============================================================
# extrinsics_pipeline — ExtrinsicsOptimizer and pipeline helpers
# ============================================================
import os, sys, time, traceback
import cv2
import numpy as np
import torch

# Constants
MIN_DEPTH_M = 0.3
MAX_DEPTH_M = 2.0
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
URDF_PATH = os.path.join(REPO_ROOT, "assets", "franka_description", "franka_panda_robotiq_2f85_og.urdf")

def _print(*args, **kwargs):
    """Print with automatic timestamp prefix."""
    if args and isinstance(args[0], str) and args[0].startswith('['):
        print(*args, flush=True, **kwargs)
    else:
        first_arg = f"[{get_time_str()}] {args[0]}" if args else f"[{get_time_str()}]"
        print(first_arg, *args[1:], flush=True, **kwargs)

class ExtrinsicsOptimizer:
    """Optimizes camera extrinsics for a single scene using robot mesh rendering and depth."""
    def __init__(self, scene_path, vggt_forward=None, robot_renderer=None, device="cuda",
                 num_frames=1, downscale_ratio=1.0, min_robot_points=2000):
        self.scene_path = scene_path; self.vggt_forward = vggt_forward
        self.robot_renderer = robot_renderer; self.device = device
        self.num_frames = num_frames; self.downscale_ratio = downscale_ratio
        self.min_robot_points = min_robot_points
        self.data_dict = {}; self.proprio_dict = {}
        self.meta = get_metadata(self.scene_path)
        self.uuid = self.meta['uuid']
        self.wrist_serial = self.meta["wrist_cam_serial"]
        self.ext_serials = [self.meta[k] for k in self.meta if k.endswith("_cam_serial") and k != "wrist_cam_serial"]
        self.optimization_metrics = {}

    def _load_scene_data(self):
        start = time.time()
        self.data_dict = gather_data_dict(self.scene_path, downscale_ratio=self.downscale_ratio,
            include_stereo=False, include_depth=False, include_wrist_cam=True, max_frames=-1)
        self.proprio_dict = gather_trajectory(self.scene_path, max_frames=-1)
        first_camera_serial = list(self.data_dict.keys())[0]
        canonical_timestamps = self.data_dict[first_camera_serial]['timestamps']
        if self.num_frames > 0 and len(canonical_timestamps) > self.num_frames:
            time_skip_ratio = max(1, len(canonical_timestamps) // self.num_frames)
            canonical_timestamps = canonical_timestamps[::time_skip_ratio]
            if len(canonical_timestamps) > self.num_frames:
                canonical_timestamps = canonical_timestamps[:self.num_frames]
        self.T = len(canonical_timestamps)
        self.data_dict = filter_by_timestamps(self.data_dict, canonical_timestamps, self.scene_path, is_proprio=False)
        self.proprio_dict = filter_by_timestamps(self.proprio_dict, canonical_timestamps, self.scene_path, is_proprio=True)
        flipped_wrist_rgb = [cv2.rotate(frame, cv2.ROTATE_180) for frame in self.data_dict[self.wrist_serial]['rgb']]
        self.data_dict[self.wrist_serial]['rgb'] = flipped_wrist_rgb
        self.T = len(self.proprio_dict['joint_positions'])
        for camera_serial in self.data_dict:
            assert 'intrinsic' in self.data_dict[camera_serial]
            self.data_dict[camera_serial]['measured_intrinsics'] = self.data_dict[camera_serial]['intrinsic'].copy()
            del self.data_dict[camera_serial]['intrinsic']
        robot_serial = get_robot_serial(self.scene_path)
        self.T_gripper_wrist = get_robot_transform(robot_serial=robot_serial)
        _print(f"Loaded data for {len(self.data_dict)} cameras, {self.T} frames (time taken: {time.time()-start:.2f}s)")

    def _compute_T_base_wrist(self, frame_idx):
        T_gripper_base_quat = self.proprio_dict['gripper_pose'][frame_idx]
        T_gripper_base = convert_pose_quat2mat(T_gripper_base_quat[None])[0]
        T_base_gripper = np.linalg.inv(T_gripper_base)
        T_base_wrist = self.T_gripper_wrist @ T_base_gripper
        flip_transform = np.array([[-1,0,0,0],[0,-1,0,0],[0,0,1,0],[0,0,0,1]], dtype=np.float64)
        return flip_transform @ T_base_wrist

    def _compute_frame_camera_loss(self, world_pts, optimized_T, depth_tensor, intrinsic, H, W, dedup_threshold):
        device = world_pts.device
        N_points = world_pts.shape[0]
        zero_loss = torch.tensor(0.0, device=device)
        zero_count = torch.tensor(0, device=device)
        ones = torch.ones((N_points, 1), device=device, dtype=torch.float32)
        pts_h = torch.cat([world_pts, ones], dim=1)
        pts_cam = torch.mm(pts_h, optimized_T.T)
        z = pts_cam[:, 2]
        valid_depth = z > 0
        if not torch.any(valid_depth): return zero_loss, zero_count
        pts_cam_valid = pts_cam[valid_depth]; z_valid = z[valid_depth]
        proj_h = torch.mm(pts_cam_valid[:, :3], intrinsic.T)
        xy = proj_h[:, :2] / (proj_h[:, 2:3] + 1e-8)
        if not isinstance(H, torch.Tensor): H = torch.tensor(H, device=device, dtype=torch.float32)
        if not isinstance(W, torch.Tensor): W = torch.tensor(W, device=device, dtype=torch.float32)
        valid_bounds = (xy[:,0] >= 0) & (xy[:,0] < W) & (xy[:,1] >= 0) & (xy[:,1] < H)
        if not torch.any(valid_bounds): return zero_loss, zero_count
        xy_valid = xy[valid_bounds]; z_final = z_valid[valid_bounds]
        if dedup_threshold > 0:
            unique_indices = deduplicate_coordinates(xy_valid, threshold_pixels=dedup_threshold)
            if len(unique_indices) == 0: return zero_loss, zero_count
            xy_final = xy_valid[unique_indices]; z_final = z_final[unique_indices]
        else:
            xy_final = xy_valid
        sampled_depth = sample_depth_with_grid_sample(depth_tensor, xy_final)
        d_pred = z_final; d_gt = sampled_depth
        valid_range = (d_gt >= MIN_DEPTH_M) & (d_gt <= MAX_DEPTH_M)
        if not torch.any(valid_range): return zero_loss, zero_count
        d_pred_f = d_pred[valid_range]; d_gt_f = d_gt[valid_range]
        frame_camera_loss = torch.abs(d_gt_f - d_pred_f).mean()
        if torch.isfinite(frame_camera_loss):
            return frame_camera_loss, torch.tensor(d_pred_f.shape[0], device=device)
        return zero_loss, zero_count

    def _optimize_all_cameras_jointly(self, camera_data_dict, initial_extrinsics_dict,
                                      max_iterations, learning_rate, translation_scale, rotation_scale,
                                      dedup_threshold=0.1, optimizer_type="adam"):
        ext_camera_serials = [s for s in camera_data_dict.keys() if s != self.wrist_serial]
        n_ext_cameras = len(ext_camera_serials)
        translation_params_norm = torch.zeros(n_ext_cameras, 3, device=self.device, dtype=torch.float32, requires_grad=True)
        rotation_params_norm = torch.zeros(n_ext_cameras, 3, device=self.device, dtype=torch.float32, requires_grad=True)
        if optimizer_type.lower() == "lbfgs":
            optimizer = torch.optim.LBFGS([translation_params_norm, rotation_params_norm], lr=learning_rate, max_iter=3, tolerance_grad=1e-5, tolerance_change=1e-7)
        else:
            optimizer = torch.optim.Adam([translation_params_norm, rotation_params_norm], lr=learning_rate, eps=1e-6, weight_decay=0.0)
        extrinsic_init_tensor = torch.zeros((n_ext_cameras, 4, 4), dtype=torch.float32, device=self.device)
        camera_to_idx = {cam: i for i, cam in enumerate(ext_camera_serials)}
        for i, cam in enumerate(ext_camera_serials):
            extrinsic_init_tensor[i] = torch.as_tensor(initial_extrinsics_dict[cam], dtype=torch.float32, device=self.device)
        all_world_points = []
        for frame_idx in range(self.num_frames):
            jp = self.proprio_dict['joint_positions'][frame_idx]
            gp = self.proprio_dict['gripper_positions'][frame_idx]
            fk_result = self.robot_renderer._get_forward_kinematics(jp, gp)
            world_pts = self.robot_renderer._get_world_points(fk_result)
            all_world_points.append(world_pts)
        all_world_points = torch.stack(all_world_points, dim=0)
        first_depth = camera_data_dict[ext_camera_serials[0]]['depth'][0]
        H, W = first_depth.shape
        frame_groups = {}
        for frame_idx in range(all_world_points.shape[0]):
            frame_groups[frame_idx] = []
            for cam_serial in ext_camera_serials:
                depth_tensor = torch.as_tensor(camera_data_dict[cam_serial]['depth'][frame_idx].copy(), dtype=torch.float32, device=self.device)
                intrinsic_tensor = torch.as_tensor(camera_data_dict[cam_serial]['intrinsic'], dtype=torch.float32, device=self.device)
                frame_groups[frame_idx].append((camera_to_idx[cam_serial], depth_tensor, intrinsic_tensor))
        per_camera_losses_history = {cam: [] for cam in ext_camera_serials}
        final_per_camera_point_counts = {cam: [] for cam in ext_camera_serials}
        total_valid_points = 0
        def compute_loss_vectorized():
            nonlocal total_valid_points, per_camera_losses_history, final_per_camera_point_counts
            optimizer.zero_grad()
            translation_params = translation_params_norm * translation_scale
            rotation_params = rotation_params_norm * rotation_scale
            optimized_extrinsics = torch.zeros_like(extrinsic_init_tensor)
            for i in range(n_ext_cameras):
                delta_T = pose_6dof_to_matrix(torch.cat([translation_params[i], rotation_params[i]]))
                optimized_extrinsics[i] = delta_T @ extrinsic_init_tensor[i]
            all_losses = []; total_valid_points = 0
            frame_camera_point_counts = {}
            per_camera_losses_this_iter = {cam: [] for cam in ext_camera_serials}
            for frame_idx, camera_data_list in frame_groups.items():
                world_pts = all_world_points[frame_idx]
                for cam_idx, depth_tensor, intrinsic in camera_data_list:
                    optimized_T = optimized_extrinsics[cam_idx]
                    loss, count = self._compute_frame_camera_loss(world_pts, optimized_T, depth_tensor, intrinsic, H, W, dedup_threshold)
                    valid_count = count.item(); cam_serial = ext_camera_serials[cam_idx]
                    frame_camera_point_counts[(frame_idx, cam_serial)] = valid_count
                    if valid_count > 0:
                        all_losses.append(loss); total_valid_points += valid_count
                        per_camera_losses_this_iter[cam_serial].append(loss.item())
            for (frame_idx, cam_serial), point_count in frame_camera_point_counts.items():
                if point_count < self.min_robot_points:
                    raise RobotVisibilityError(f"Camera {cam_serial} at frame {frame_idx} sees {point_count} robot points (< {self.min_robot_points})", error_type="insufficient_robot_points_per_frame_camera")
            data_loss = torch.stack(all_losses).mean() if all_losses else torch.tensor(1e6, device=self.device, dtype=torch.float32)
            total_loss = data_loss
            for cam in ext_camera_serials:
                per_camera_losses_history[cam].append(np.mean(per_camera_losses_this_iter[cam]) if per_camera_losses_this_iter[cam] else 0.0)
            for cam in ext_camera_serials:
                final_per_camera_point_counts[cam] = [frame_camera_point_counts.get((fi, cam), 0) for fi in range(self.num_frames)]
            if torch.isfinite(total_loss): total_loss.backward()
            else: _print("    Warning: Loss is not finite!")
            return total_loss
        initial_loss = compute_loss_vectorized()
        loss_history = []
        for iteration in range(max_iterations):
            if not (torch.isfinite(translation_params_norm).all() and torch.isfinite(rotation_params_norm).all()):
                _print(f"    iter {iteration:03d} | NaN/inf in params, stopping"); break
            loss_value = optimizer.step(compute_loss_vectorized)
            current_loss = loss_value.item() if torch.isfinite(loss_value) else float('inf')
            loss_history.append(current_loss)
            if iteration % 50 == 0 or iteration < 5 or iteration == max_iterations - 1:
                avg_points_per_frame = total_valid_points / max(1, self.num_frames * n_ext_cameras)
                _print(f"    iter {iteration:03d} | loss {current_loss:.6f} | robot_pts/cam-frame {avg_points_per_frame:.1f}")
        optimized_extrinsics_dict = {}
        for i, cam_serial in enumerate(ext_camera_serials):
            final_pose = torch.cat([translation_params_norm[i].detach() * translation_scale, rotation_params_norm[i].detach() * rotation_scale])
            optimized_extrinsics_dict[cam_serial] = (pose_6dof_to_matrix(final_pose) @ extrinsic_init_tensor[i]).cpu().numpy()
        all_point_counts = [c for cam in ext_camera_serials for c in final_per_camera_point_counts[cam]]
        optimization_metrics = {
            "initial_loss": float(initial_loss.item()), "final_loss": float(loss_history[-1] if loss_history else 0),
            "min_robot_points_threshold": self.min_robot_points,
            "average_robot_points": float(np.mean(all_point_counts)) if all_point_counts else 0,
        }
        return optimized_extrinsics_dict, optimization_metrics

    def add_depth_valid_masks(self):
        for camera_serial in self.data_dict:
            for key in list(self.data_dict[camera_serial].keys()):
                if key.endswith('_depth'):
                    depth_frames = self.data_dict[camera_serial][key]
                    valid_mask = np.isfinite(depth_frames) & (depth_frames >= MIN_DEPTH_M) & (depth_frames <= MAX_DEPTH_M)
                    self.data_dict[camera_serial][f'{key}_valid_mask'] = valid_mask

    def _prepare_vggt_input(self):
        aggregator_images, aggregator_cam_ids, aggregator_frame_idx = [], [], []
        first_ext_serial = self.ext_serials[0]
        for i in range(len(self.data_dict[first_ext_serial]['rgb'])):
            aggregator_images.append(self.data_dict[first_ext_serial]['rgb'][i])
            aggregator_cam_ids.append(first_ext_serial); aggregator_frame_idx.append(i)
        for i in range(len(self.data_dict[self.wrist_serial]['rgb'])):
            aggregator_images.append(self.data_dict[self.wrist_serial]['rgb'][i])
            aggregator_cam_ids.append(self.wrist_serial); aggregator_frame_idx.append(i)
        for cam_serial in self.ext_serials[1:]:
            for i in range(len(self.data_dict[cam_serial]['rgb'])):
                aggregator_images.append(self.data_dict[cam_serial]['rgb'][i])
                aggregator_cam_ids.append(cam_serial); aggregator_frame_idx.append(i)
        return aggregator_images, aggregator_cam_ids, aggregator_frame_idx

    def _convert_vggt_to_base_frame(self, vggt_extrinsics, aggregator_cam_ids, aggregator_frame_idx):
        result = {}
        wrist_indices = [idx for idx, cser in enumerate(aggregator_cam_ids) if cser == self.wrist_serial]
        assert len(wrist_indices) > 0
        T_base_ext0_estimates = []
        for wrist_idx in wrist_indices:
            frame_idx = aggregator_frame_idx[wrist_idx]
            T_base_wrist_t = self._compute_T_base_wrist(frame_idx)
            T_ext0_wrist_t = vggt_extrinsics[wrist_idx]
            T_base_ext0_estimates.append(np.linalg.inv(T_ext0_wrist_t) @ T_base_wrist_t)
        T_base_ext0_avg = average_poses(T_base_ext0_estimates) if len(T_base_ext0_estimates) > 1 else T_base_ext0_estimates[0]
        result[self.ext_serials[0]] = T_base_ext0_avg
        for cam_serial in self.ext_serials[1:]:
            indices = [idx for idx, cser in enumerate(aggregator_cam_ids) if cser == cam_serial]
            T_list = [vggt_extrinsics[idx] for idx in indices]
            T_avg = average_poses(T_list) if len(T_list) > 1 else T_list[0]
            result[cam_serial] = T_avg @ T_base_ext0_avg
        return result

    def _filter_frames_by_robot_visibility(self, extrinsics_dict):
        start = time.time(); valid_frame_indices = []
        for frame_idx in range(self.T):
            frame_valid = True
            jp = self.proprio_dict['joint_positions'][frame_idx]
            gp = self.proprio_dict['gripper_positions'][frame_idx]
            fk_result = self.robot_renderer._get_forward_kinematics(jp, gp)
            world_pts = self.robot_renderer._get_world_points(fk_result)
            for camera_serial in self.data_dict:
                if camera_serial == self.wrist_serial or camera_serial not in extrinsics_dict: continue
                extrinsic = extrinsics_dict[camera_serial]
                depth_frame = self.data_dict[camera_serial]['stereo_depth'][frame_idx]
                intrinsic = self.data_dict[camera_serial]['stereo_intrinsics']
                _, valid_count_t = self._compute_frame_camera_loss(
                    torch.as_tensor(world_pts, dtype=torch.float32, device=self.device),
                    torch.as_tensor(extrinsic, dtype=torch.float32, device=self.device),
                    torch.as_tensor(depth_frame, dtype=torch.float32, device=self.device),
                    torch.as_tensor(intrinsic, dtype=torch.float32, device=self.device),
                    *depth_frame.shape, dedup_threshold=0.1
                )
                if valid_count_t.item() < self.min_robot_points:
                    _print(f"    Frame {frame_idx}: Camera {camera_serial} sees only {valid_count_t.item()} robot points, removing frame")
                    frame_valid = False; break
            if frame_valid: valid_frame_indices.append(frame_idx)
        _print(f"Kept {len(valid_frame_indices)}/{self.T} frames after robot visibility filtering (time taken: {time.time()-start:.2f}s)")
        if len(valid_frame_indices) == 0:
            raise RobotVisibilityError(f"No frames have sufficient robot visibility.", error_type="no_valid_frames_after_filtering")
        return valid_frame_indices

    def _update_data_for_valid_frames(self, valid_frame_indices):
        if len(valid_frame_indices) == self.T: return
        _print(f"Updating data structures for {len(valid_frame_indices)} valid frames...")
        for key in self.proprio_dict:
            if isinstance(self.proprio_dict[key], list):
                self.proprio_dict[key] = [self.proprio_dict[key][i] for i in valid_frame_indices]
            elif isinstance(self.proprio_dict[key], np.ndarray) and len(self.proprio_dict[key]) == self.T:
                self.proprio_dict[key] = self.proprio_dict[key][valid_frame_indices]
        for camera_serial in self.data_dict:
            for key in self.data_dict[camera_serial]:
                if isinstance(self.data_dict[camera_serial][key], list) and len(self.data_dict[camera_serial][key]) == self.T:
                    self.data_dict[camera_serial][key] = [self.data_dict[camera_serial][key][i] for i in valid_frame_indices]
                elif isinstance(self.data_dict[camera_serial][key], np.ndarray) and len(self.data_dict[camera_serial][key]) == self.T:
                    self.data_dict[camera_serial][key] = self.data_dict[camera_serial][key][valid_frame_indices]
        self.T = len(valid_frame_indices); self.num_frames = min(self.num_frames, self.T)

    def vggt_estimation(self):
        assert self.vggt_forward is not None
        start = time.time()
        aggregator_images, aggregator_cam_ids, aggregator_frame_idx = self._prepare_vggt_input()
        with stage_vggt_images(aggregator_images) as temp_image_paths:
            vggt_result = self.vggt_forward(temp_image_paths)
        vggt_extrinsics_4x4 = vggt_result['extrinsics_4x4']
        vggt_intrinsics = vggt_result['intrinsics']
        base_frame_extrinsics = self._convert_vggt_to_base_frame(vggt_extrinsics_4x4, aggregator_cam_ids, aggregator_frame_idx)
        for idx, cam_serial in enumerate(aggregator_cam_ids):
            if cam_serial == self.wrist_serial: continue
            self.data_dict[cam_serial]['vggt_extrinsics'] = base_frame_extrinsics[cam_serial]
            self.data_dict[cam_serial]['vggt_intrinsics'] = vggt_intrinsics[idx]
        _print(f"VGGT estimation completed (time taken: {time.time()-start:.2f}s)")
        valid_frame_indices = self._filter_frames_by_robot_visibility(base_frame_extrinsics)
        self._update_data_for_valid_frames(valid_frame_indices)

    def _collect_optimizer_inputs(self, depth_key, intrinsics_key, extrinsics_key):
        camera_data_dict = {}; initial_extrinsics_dict = {}
        for camera_serial in self.data_dict:
            if camera_serial == self.wrist_serial: continue
            camera_data_dict[camera_serial] = {
                'rgb': self.data_dict[camera_serial]['rgb'],
                'depth': self.data_dict[camera_serial][depth_key],
                'depth_valid_mask': self.data_dict[camera_serial][f'{depth_key}_valid_mask'],
                'intrinsic': self.data_dict[camera_serial][intrinsics_key],
            }
            initial_extrinsics_dict[camera_serial] = self.data_dict[camera_serial][extrinsics_key]
        return camera_data_dict, initial_extrinsics_dict

    def optimize_extrinsics(self, max_iterations=100, learning_rate=0.001, translation_scale=0.01,
                            rotation_scale=np.pi/180.0, dedup_threshold=0.5, optimizer_type="adam"):
        start = time.time()
        assert self.robot_renderer is not None
        camera_data_dict, initial_extrinsics_dict = self._collect_optimizer_inputs('stereo_depth', 'stereo_intrinsics', 'vggt_extrinsics')
        optimized_extrinsics_dict, optimization_metrics = self._optimize_all_cameras_jointly(
            camera_data_dict=camera_data_dict, initial_extrinsics_dict=initial_extrinsics_dict,
            max_iterations=max_iterations, learning_rate=learning_rate,
            translation_scale=translation_scale, rotation_scale=rotation_scale,
            dedup_threshold=dedup_threshold, optimizer_type=optimizer_type)
        for cam_serial, opt_extrinsic in optimized_extrinsics_dict.items():
            self.data_dict[cam_serial]['optimized_extrinsics'] = opt_extrinsic
        self.optimization_metrics = optimization_metrics
        _print(f"Joint extrinsics optimization completed (time taken: {time.time()-start:.2f}s)")

    def process(self, output_dir=None, max_iterations=100, learning_rate=0.001, translation_scale=0.01,
                rotation_scale=np.pi/180.0, dedup_threshold=1.0, optimizer_type="adam", min_robot_points=None):
        if min_robot_points is not None: self.min_robot_points = min_robot_points
        self._load_scene_data()
        if self.T < self.num_frames:
            raise ValueError(f"too few frames in scene {self.scene_path}, T={self.T}, num_frames={self.num_frames}")
        if output_dir is None: raise ValueError("output_dir must be provided to load precomputed depth")
        load_precomputed_depth(output_dir=output_dir, uuid=self.uuid, data_dict=self.data_dict, wrist_serial=self.wrist_serial, log_fn=_print)
        self.vggt_estimation()
        self.add_depth_valid_masks()
        self.optimize_extrinsics(max_iterations=max_iterations, learning_rate=learning_rate,
            translation_scale=translation_scale, rotation_scale=rotation_scale,
            dedup_threshold=dedup_threshold, optimizer_type=optimizer_type)
        if output_dir is not None:
            write_camera_results(output_dir=output_dir, uuid=self.uuid, scene_path=self.scene_path,
                data_dict=self.data_dict, wrist_serial=self.wrist_serial,
                optimization_metrics=self.optimization_metrics, error_info=None, log_fn=_print)

def determine_error_stage():
    exc_type, exc_value, exc_traceback = sys.exc_info()
    stack_str = str(traceback.extract_tb(exc_traceback)) if exc_traceback else str(traceback.extract_stack())
    if '_optimize_all_cameras_jointly' in stack_str or 'optimize_extrinsics' in stack_str: return "extrinsics_optimization"
    elif '_filter_frames_by_robot_visibility' in stack_str or 'vggt_estimation' in stack_str: return "vggt_estimation"
    elif '_load_scene_data' in stack_str: return "data_loading"
    return "unknown"

def process_single_scene(scene_path, output_dir, vggt_forward, robot_renderer, num_frames, downscale_ratio,
                         min_robot_points, max_iterations, lr, translation_scale, rotation_scale,
                         dedup_threshold, optimizer, debug):
    """Process a single scene for extrinsics optimization."""
    scene_optimizer = None
    try:
        uuid = get_uuid(scene_path)
        if output_dir is None: raise ValueError("output_dir must be provided")
        depth_exists, h5_path, error_msg = check_precomputed_depth_exists(scene_path, output_dir, uuid)
        if not depth_exists: raise FileNotFoundError(error_msg)
        _print(f"Verified precomputed depth file: {h5_path}")
        if output_dir is not None:
            results_exist, json_path, error_msg = check_camera_results_exist(scene_path, output_dir, uuid)
            if results_exist:
                _print(f"Camera results already exist: {json_path}, skipping")
                return True
        start = time.time()
        scene_optimizer = ExtrinsicsOptimizer(scene_path=scene_path, vggt_forward=vggt_forward,
            robot_renderer=robot_renderer, device=DEVICE, num_frames=num_frames,
            downscale_ratio=downscale_ratio, min_robot_points=min_robot_points)
        scene_optimizer.process(output_dir=output_dir, max_iterations=max_iterations, learning_rate=lr,
            translation_scale=translation_scale, rotation_scale=rotation_scale,
            dedup_threshold=dedup_threshold, optimizer_type=optimizer)
        _print(f"Successfully processed scene: {scene_optimizer.uuid} (time taken: {time.time()-start:.2f}s)")
        return True
    except RobotVisibilityError as e:
        _print(f"Insufficient robot visibility for scene {scene_path}: {e}")
        if output_dir is not None and scene_optimizer is not None:
            write_camera_results(output_dir=output_dir, uuid=scene_optimizer.uuid, scene_path=scene_path,
                data_dict=scene_optimizer.data_dict, wrist_serial=scene_optimizer.wrist_serial,
                optimization_metrics=scene_optimizer.optimization_metrics,
                error_info={"error_type": getattr(e, 'error_type', 'robot_visibility_error'), "error_message": str(e), "stage": determine_error_stage(), "min_robot_points_threshold": min_robot_points},
                log_fn=_print)
        return False
    except Exception as e:
        if 'Precomputed depth file not found' in str(e):
            _print(f"Precomputed depth file not found, skipping")
            return False
        error_msg = f"Error processing scene {scene_path}: {e}"
        if debug:
            _print(error_msg); traceback.print_exc(); raise
        _print(f"{error_msg}, skipping")
        return False

# Helper: average_poses (used by _convert_vggt_to_base_frame)
def average_poses(pose_list):
    """Average a list of 4x4 transformation matrices."""
    translations = np.stack([p[:3, 3] for p in pose_list])
    avg_translation = translations.mean(axis=0)
    from scipy.spatial.transform import Rotation as R
    rotations = R.from_matrix([p[:3, :3] for p in pose_list])
    avg_rotation = rotations.mean()
    result = np.eye(4); result[:3, :3] = avg_rotation.as_matrix(); result[:3, 3] = avg_translation
    return result

print("✅ ExtrinsicsOptimizer + process_single_scene defined")
print(f"Device: {DEVICE}")
print(f"URDF path: {URDF_PATH}")

### Compute Extrinsics — Configuration

Edit the variables below, then run the remaining cells.

In [ ]:
# ============================================================
# Configuration — edit these values
# ============================================================
EXT_OUTPUT_DIR        = OUTPUT_DIR                # same as compute_depth OUTPUT_DIR
EXT_INPUT_FILE        = INPUT_FILE                # same input file
SCENE                 = None                      # single scene path (set None to use EXT_INPUT_FILE)
EXT_RANK              = 0
EXT_WORLD_SIZE        = 1
DEBUG                 = False

# Processing parameters
NUM_FRAMES            = 10
DOWNSCALE             = 0.5
MAX_ITERATIONS        = 2000
LR                    = 0.05
TRANSLATION_SCALE     = 0.01
ROTATION_SCALE        = np.deg2rad(0.05)     # radians (~0.05 degrees)
DEDUP_THRESHOLD       = 0.5
MIN_ROBOT_POINTS      = 1000
VGGT_MODEL_PATH       = os.path.join(REPO_ROOT, "checkpoints", "vggt", "model.pt")
OPTIMIZER             = "adam"               # "adam" or "lbfgs"
EXT_ALLOW_GCS_STREAMING = True               # True: stream from GCS

print(f"EXT_OUTPUT_DIR: {EXT_OUTPUT_DIR}")
print(f"VGGT_MODEL_PATH: {VGGT_MODEL_PATH}")

In [ ]:
# Prepare scene paths
assert SCENE or EXT_INPUT_FILE, "Either SCENE or EXT_INPUT_FILE must be provided"

if EXT_INPUT_FILE:
    with open(EXT_INPUT_FILE, 'r') as f:
        ext_all_paths = [line.strip() for line in f if line.strip()]

    # Prepend GCS prefix if paths are relative
    if SCENE_PATH_PREFIX:
        ext_all_paths = [
            (SCENE_PATH_PREFIX + p) if not p.startswith('gs://') and not p.startswith('/') else p
            for p in ext_all_paths
        ]

    random.seed(42)
    random.shuffle(ext_all_paths)

    if EXT_WORLD_SIZE <= 0:
        raise ValueError(f"EXT_WORLD_SIZE must be >= 1, got {EXT_WORLD_SIZE}")
    if not (0 <= EXT_RANK < EXT_WORLD_SIZE):
        raise ValueError(f"EXT_RANK must be in [0, EXT_WORLD_SIZE), got rank={EXT_RANK}, world_size={EXT_WORLD_SIZE}")
    ext_paths_to_process = ext_all_paths[EXT_RANK::EXT_WORLD_SIZE]
else:
    ext_all_paths = [SCENE]
    ext_paths_to_process = ext_all_paths

enforce_gcs_cache_policy(
    ext_all_paths,
    stage_name="compute_extrinsics",
    require_cache=True,
    allow_streaming=EXT_ALLOW_GCS_STREAMING,
)

_print(f"Rank {EXT_RANK}/{EXT_WORLD_SIZE}: Processing {len(ext_paths_to_process)}/{len(ext_all_paths)} scenes")

In [ ]:
# Initialize VGGT model and robot renderer
if len(ext_paths_to_process) == 0:
    _print(f"No scenes to process for rank {EXT_RANK}")
else:
    # Load VGGT
    from vggt.models.vggt import VGGT
    assert os.path.exists(VGGT_MODEL_PATH), f"VGGT model file not found: {VGGT_MODEL_PATH}"
    vggt_model = VGGT()
    vggt_model.load_state_dict(torch.load(VGGT_MODEL_PATH, map_location=DEVICE))
    vggt_model.to(DEVICE)
    vggt_model.eval()
    vggt_forward = VGGTForwardPass(vggt_model, device=DEVICE)
    _print(f"VGGT model loaded successfully from {VGGT_MODEL_PATH}")

    # Initialize robot renderer
    robot_renderer = RobotMeshRenderer(URDF_PATH, device=DEVICE, total_samples=25000)

In [ ]:
# Process all scenes
total_processed = 0
total_successful = 0
total_skipped = 0

for scene_path in tqdm(ext_paths_to_process, desc=f"Rank {EXT_RANK}/{EXT_WORLD_SIZE}"):
    total_processed += 1

    success = process_single_scene(
        scene_path=scene_path,
        output_dir=EXT_OUTPUT_DIR,
        vggt_forward=vggt_forward,
        robot_renderer=robot_renderer,
        num_frames=NUM_FRAMES,
        downscale_ratio=DOWNSCALE,
        min_robot_points=MIN_ROBOT_POINTS,
        max_iterations=MAX_ITERATIONS,
        lr=LR,
        translation_scale=TRANSLATION_SCALE,
        rotation_scale=ROTATION_SCALE,
        dedup_threshold=DEDUP_THRESHOLD,
        optimizer=OPTIMIZER,
        debug=DEBUG
    )

    if success:
        total_successful += 1
    else:
        total_skipped += 1

_print(f"\nProcessing completed!")
_print(f"  Total scenes processed: {total_processed}")
_print(f"  Successful: {total_successful}")
_print(f"  Skipped: {total_skipped}")
_print(f"  Success rate: {100*total_successful/total_processed:.1f}%")